# Token-Efficient RAG: Reducing Context Size and Latency Through Reranking and Contextual Compression

**Research question.** *To what extent can reranking and extractive contextual compression reduce context-token usage and response latency in a local RAG system while maintaining answer quality?*

This notebook is the complete, self-contained implementation of the experiment. It covers data preparation, both RAG pipelines, the test suite, execution, analysis, charts and the generated report.
Every number it reports is computed from the recorded results. No results are estimated, and answer-quality scores are **not** assigned automatically.

### Experimental design

| | **A. Baseline RAG** | **B. Token-efficient RAG** |
|---|---|---|
| Retrieval | FAISS top-5 | FAISS top-10 |
| Reranking | — | cross-encoder `ms-marco-MiniLM-L-6-v2` → top-3 |
| Context sent to LLM | full text of the 5 chunks | extractive sentence compression of the 3 chunks (≤ 700 tokens) |
| Shared by both | documents, 500/80 chunking, `all-MiniLM-L6-v2` embeddings, one FAISS index, Gemma 3 4B Instruct Q4_K_M in LM Studio, identical system prompt, template and generation parameters (temperature 0) | |

```
A: question → embed → FAISS top-5 ───────────────────────────────────────→ full chunks ──→ Gemma 3 4B → answer
B: question → embed → FAISS top-10 → cross-encoder rerank → top-3 → keep relevant sentences ──→ Gemma 3 4B → answer
```

### How to run
1. **Laptop:** start the LM Studio server (`http://localhost:1234/v1`) with **gemma-3-4b-it** loaded; this is only needed for new LLM runs. **Google Colab:** select a GPU runtime and run all cells; section 0 sets up an equivalent server automatically, and the default mode there is `full`.
2. Set `RUN_MODE` in the first code cell, or run `run_notebook.bat`, which asks which mode to use.
3. Choose **Kernel → Restart & Run All**.

| `RUN_MODE` | What happens |
|---|---|
| `"analyze_existing"` | No LLM calls. Recomputes all metrics, checks, charts and the report from `results/experiment_results.csv`. |
| `"smoke"` | Preflight, integration tests and the 2-question smoke test with its gate. |
| `"full"` | Preflight, then tests, then the smoke test and gate, then **N_RUNS = 3** full passes (144 LLM calls), then analysis, report and self-check. Earlier results are moved to `results/archive/`, never deleted. |
| `"resume"` | Continues an interrupted full run; only missing or failed pipeline runs are repeated. |

### Contents
0. Colab setup · 1. Configuration · 2. Chunking · 3. Token counting · 4. Embeddings & FAISS · 5. Extractive compression · 6. LM Studio client · 7. The two pipelines ·
8. Preflight · 9. Integration tests · 10. Experiment runner · 11. Smoke gate · 12. Analysis & charts · 13. Evidence coverage · 14. Latency decomposition ·
15. Replication vs pilot run · 16. Report builder · 17. Offline test suite · 18. Final self-check · 19. **Run** · 20. Results & visualisations · 21. Export (Colab) · 22. Author's notes

## 0. Google Colab setup (runs only in Colab)
Colab cannot reach LM Studio on a laptop. When the notebook runs in Colab, the next two cells therefore:
1. unpack the project files (documents, questions and the preserved pilot run) that are embedded in this notebook;
2. install the packages;
3. build llama.cpp's `llama-server` (with CUDA if a GPU is attached);
4. download the **same** GGUF model file (`Aldaris/gemma-3-4b-it-Q4_K_M-GGUF`) and record its SHA-256;
5. serve it at `http://localhost:1234/v1` under the model ID `gemma-3-4b-it`.

All pipeline settings are unchanged. The runtime details are recorded in `experiment_metadata.json` under `llm_runtime`.
Outside Colab, both cells do nothing.

In [ ]:
# Embedded project files (data/ and results/pilot_run_2026-09-28/), base64-encoded zip - used only in Colab
PROJECT_FILES_B64 = "..."

In [ ]:
# ===================== GOOGLE COLAB SETUP (skipped automatically outside Colab) =====================
# Colab cannot reach LM Studio on a laptop, so here the SAME model file (Aldaris/gemma-3-4b-it-Q4_K_M-GGUF)
# is served by llama.cpp's OpenAI-compatible `llama-server` on http://localhost:1234/v1 under the model ID
# "gemma-3-4b-it". Every pipeline setting below is unchanged; only the LLM runtime/hardware differs
# (recorded in experiment_metadata.json -> llm_runtime).
import sys, os, subprocess, base64, io, zipfile, json, time, hashlib, glob, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUNTIME_INFO = {"name": "LM Studio (desktop app)", "api": "OpenAI-compatible /v1 at http://localhost:1234/v1"}

def _sh(cmd, log=None, check=True):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    if log:
        Path(log).write_text(r.stdout + r.stderr)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout.strip()

if IN_COLAB:
    t0 = time.time()
    ROOT_DIR = Path("/content/Token_Efficient_RAG")
    if not (ROOT_DIR / "data" / "documents").is_dir():
        ROOT_DIR.mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_FILES_B64))).extractall(ROOT_DIR)
    os.chdir(ROOT_DIR)
    os.environ["RAG_PROJECT_ROOT"] = str(ROOT_DIR)
    os.environ.setdefault("RAG_RUN_MODE", "full")

    # 1) Python packages (Colab already ships torch, transformers, pandas, matplotlib, scipy)
    _sh([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "faiss-cpu", "openai",
         "python-dotenv", "tiktoken", "psutil", "huggingface_hub"])

    # 2) Build llama.cpp's llama-server (CUDA if a GPU is attached), recording the exact commit
    has_gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
    if not Path("/content/llama.cpp/build/bin/llama-server").exists():
        _sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
        flags = "-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native" if has_gpu else ""
        _sh(f"cmake -S /content/llama.cpp -B /content/llama.cpp/build {flags} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release",
            log="/content/llama_cmake.log")
        _sh("cmake --build /content/llama.cpp/build --config Release -j 8 --target llama-server", log="/content/llama_build.log")
    commit = _sh("git -C /content/llama.cpp rev-parse HEAD")

    # 3) Download the same GGUF model file used on the laptop and fingerprint it
    from huggingface_hub import hf_hub_download, list_repo_files
    REPO = "Aldaris/gemma-3-4b-it-Q4_K_M-GGUF"
    gguf_name = [f for f in list_repo_files(REPO) if f.lower().endswith(".gguf")][0]
    gguf_path = hf_hub_download(REPO, gguf_name)
    h = hashlib.sha256()
    with open(gguf_path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)

    # 4) Start llama-server on port 1234 with the model ID the notebook expects
    server_args = ["-m", gguf_path, "--alias", "gemma-3-4b-it", "--host", "127.0.0.1", "--port", "1234",
                   "-c", "8192", "-np", "1", "--jinja"] + (["-ngl", "999"] if has_gpu else [])
    try:
        urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=2)
        print("llama-server already running")
    except Exception:
        subprocess.Popen(["/content/llama.cpp/build/bin/llama-server"] + server_args,
                         stdout=open("/content/llama_server.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(180):
            try:
                urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=2); break
            except Exception:
                time.sleep(2)
        else:
            print(Path("/content/llama_server.log").read_text()[-3000:])
            raise RuntimeError("llama-server did not start")
    gpu = _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False) if has_gpu else None
    RUNTIME_INFO = {
        "name": "llama.cpp llama-server (Google Colab)",
        "api": "OpenAI-compatible /v1 at http://localhost:1234/v1",
        "llama_cpp_commit": commit,
        "build": "CUDA (GGML_CUDA=ON)" if has_gpu else "CPU",
        "server_args": [a if a != gguf_path else Path(gguf_path).name for a in server_args],
        "model_file": gguf_name, "model_repository": REPO, "model_file_sha256": h.hexdigest(),
        "model_file_bytes": Path(gguf_path).stat().st_size, "gpu": gpu,
        "note": "Pilot run used LM Studio on a Windows laptop; latency is not comparable across the two environments.",
    }
    print(json.dumps(RUNTIME_INFO, indent=2))
    print(f"Colab setup finished in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ===================== RUN SETTINGS =====================
import os, sys, types
from pathlib import Path

# "analyze_existing" : no LLM calls - recompute metrics, checks, charts and report from results/
# "smoke"            : preflight + integration tests + 2-question smoke test (results/smoke_test/)
# "full"             : preflight + tests + smoke test + gate + full experiment (N_RUNS passes) + analysis
# "resume"           : continue an interrupted full run (re-attempts only missing/failed runs)
RUN_MODE = os.environ.get("RAG_RUN_MODE", "analyze_existing")
RUN_OFFLINE_TESTS = True     # offline logic tests (stand-in models; no downloads, no LLM calls)

PROJECT_ROOT = Path(os.environ.get("RAG_PROJECT_ROOT", Path.cwd())).resolve()
assert (PROJECT_ROOT / "data" / "documents").is_dir(), f"data/documents not found under {PROJECT_ROOT}"
print("Project root :", PROJECT_ROOT)
print("Python       :", sys.version.split()[0])
print("RUN_MODE     :", RUN_MODE)

## 1. Configuration
All experimental settings are fixed here, and all of them are copied into `results/experiment_metadata.json` at run time.
Any setting whose name is not prefixed with a pipeline name is shared by both pipelines.
The compression parameters (threshold 0.0, minimum 5 segments, 700-token budget) were chosen **before** any evaluation run and were not tuned on the evaluation questions.

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:  # python-dotenv optional; env vars may already be set
    pass

ROOT = PROJECT_ROOT
DOCS_DIR = ROOT / "data" / "documents"
QUESTIONS_PATH = ROOT / "data" / "eval_questions.json"
INDEX_DIR = ROOT / "index"
RESULTS_DIR = ROOT / "results"

# ---- LLM (LM Studio, OpenAI-compatible) ----
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "lm_studio")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")  # local placeholder, not a secret
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gemma-3-4b-it")
MODEL_REPOSITORY = "Aldaris/gemma-3-4b-it-Q4_K_M-GGUF"  # as reported by the user
MODEL_QUANTIZATION = "Q4_K_M"                           # as reported by the user

# Identical generation parameters for both pipelines.
GENERATION_PARAMS = {
    "temperature": 0.0,
    "top_p": 1.0,
    "max_tokens": 512,
    "seed": 42,
}
LLM_STREAM = True           # streaming lets us measure time-to-first-token
LLM_TIMEOUT_SECONDS = 300
MAX_TRANSIENT_RETRIES = 2

# ---- Exact system instruction (identical for both pipelines) ----
SYSTEM_PROMPT = (
    "Answer only from the provided context. Do not use outside knowledge. "
    "If the answer is not contained in the context, say exactly: "
    "'I cannot find the answer in the provided documents.' "
    "Cite the relevant source chunk IDs in square brackets at the end of each answer."
)
REFUSAL_TEXT = "I cannot find the answer in the provided documents."

# User-message template (identical for both pipelines; only {context} differs).
USER_TEMPLATE = "Context:\n{context}\n\nQuestion: {question}"

# ---- Repetitions ----
# The full experiment is run N_RUNS times (complete passes over all 24 questions x 2 pipelines).
# Pipeline order within a question alternates by question and is flipped on every pass, so each
# question is run baseline-first and optimized-first equally often (for even N_RUNS) or +-1 (odd).
N_RUNS = 3

# ---- Chunking ----
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

# ---- Retrieval / reranking ----
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
DEVICE = "cpu"  # both local encoders forced to CPU so they do not compete with LM Studio for GPU
BASELINE_TOP_K = 5
OPTIMIZED_INITIAL_K = 10
OPTIMIZED_FINAL_K = 3

# ---- Extractive compression (fixed a priori; NOT tuned on evaluation questions) ----
COMPRESSION_TOKEN_TARGET = 700      # upper budget on compressed context tokens
COMPRESSION_SCORE_THRESHOLD = 0.0   # cross-encoder logit; 0.0 == sigmoid 0.5 decision boundary
COMPRESSION_MIN_SEGMENTS = 5        # always keep at least the 5 highest-scoring segments

# ---- Token counting ----
# tiktoken has no Gemma encoding, so a Gemma-3 tokenizer is used for local counts.
GEMMA_TOKENIZER_REPO = os.getenv("GEMMA_TOKENIZER_REPO", "unsloth/gemma-3-4b-it")

DOC_ID_MAP = {
    "01_employee_leave_policy.md": "LEAVE",
    "02_it_support_procedures.md": "IT",
    "03_product_return_policy.md": "RETURNS",
    "04_customer_support_escalation.md": "ESCALATION",
    "05_information_security_rules.md": "SECURITY",
    "06_remote_work_and_office_hours.md": "REMOTE",
}


# Collect the UPPER-CASE settings above into one read-only-by-convention namespace: config.X
_SKIP = {"RUN_MODE", "RUN_OFFLINE_TESTS", "PROJECT_ROOT"}
config = types.SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper() and k not in _SKIP})
print("LLM          :", config.OPENAI_MODEL, "@", config.OPENAI_BASE_URL)
print("Generation   :", config.GENERATION_PARAMS)
print("Chunking     :", config.CHUNK_SIZE, "chars /", config.CHUNK_OVERLAP, "overlap")
print("Retrieval    : baseline top-%d | optimized top-%d -> rerank -> top-%d -> compress (<=%d tokens)" % (
    config.BASELINE_TOP_K, config.OPTIMIZED_INITIAL_K, config.OPTIMIZED_FINAL_K, config.COMPRESSION_TOKEN_TARGET))
print("Repetitions  :", config.N_RUNS)

## 2. Document chunking
The documents are split into deterministic windows of at most 500 characters with 80 characters of overlap, cut at word boundaries.
Each chunk keeps its **provenance** (document ID, document name, chunk ID and character offsets), so that `text == document[char_start:char_end]` holds for every chunk.

In [ ]:
"""Deterministic character chunking with provenance (doc id, chunk id, char offsets)."""
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List



@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    doc_name: str
    chunk_index: int
    char_start: int
    char_end: int
    text: str

    def to_dict(self):
        return asdict(self)


def split_text(text: str, chunk_size: int, overlap: int):
    """Return (start, end) windows of at most chunk_size chars.

    Windows end at the last whitespace inside the window (if one exists in its
    second half) so words are not cut; each next window starts `overlap` chars
    before the previous end, moved forward to a word boundary. Leading/trailing
    whitespace is trimmed from the offsets so text == source[start:end].
    """
    assert 0 <= overlap < chunk_size
    n = len(text)
    spans = []
    start = 0
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            cut = max(text.rfind(" ", start, end), text.rfind("\n", start, end))
            if cut > start + chunk_size // 2:
                end = cut
        s, e = start, end
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        if e > s:
            spans.append((s, e))
        if end >= n:
            break
        nxt = max(end - overlap, start + 1)
        # move forward to the start of a word so chunks do not begin mid-word
        if nxt > 0 and not text[nxt - 1].isspace():
            ws = min([p for p in (text.find(" ", nxt, end), text.find("\n", nxt, end)) if p != -1], default=-1)
            if ws != -1:
                nxt = ws + 1
        start = nxt
    return spans


def load_and_chunk(docs_dir: Path = config.DOCS_DIR) -> List[Chunk]:
    chunks: List[Chunk] = []
    files = sorted(p for p in docs_dir.glob("*.md"))
    if not files:
        raise FileNotFoundError(f"No documents found in {docs_dir}")
    for path in files:
        doc_id = config.DOC_ID_MAP.get(path.name)
        if doc_id is None:
            raise KeyError(f"No doc id mapping for {path.name}")
        text = path.read_text(encoding="utf-8")
        for i, (s, e) in enumerate(split_text(text, config.CHUNK_SIZE, config.CHUNK_OVERLAP)):
            chunks.append(Chunk(
                chunk_id=f"{doc_id}-C{i:02d}", doc_id=doc_id, doc_name=path.name,
                chunk_index=i, char_start=s, char_end=e, text=text[s:e]))
    return chunks

## 3. Token counting
`tiktoken` only includes OpenAI vocabularies and cannot map Gemma, so it is **not** used for any measurement.
A Gemma-3 tokenizer (`unsloth/gemma-3-4b-it`, an ungated copy of the official tokenizer) counts context tokens for **both** pipelines.
Prompt and output tokens are taken from LM Studio's `usage` field, which uses the loaded GGUF's own tokenizer. The local count is stored alongside as a cross-check; the preflight also compares the two live.

In [ ]:
"""Consistent local token counting with a Gemma-3 tokenizer.

tiktoken only ships OpenAI encodings (cl100k/o200k, ...) and has no Gemma
vocabulary, so it is NOT used for measurements. The Gemma-3 tokenizer from
config.GEMMA_TOKENIZER_REPO (an ungated mirror of google/gemma-3-4b-it's
tokenizer) is used for ALL local context counts in BOTH pipelines.
Prompt/output token counts are additionally taken from the LM Studio server's
`usage` field (the loaded GGUF's own tokenizer) whenever it is returned.
"""
from functools import lru_cache



class GemmaTokenCounter:
    def __init__(self, repo: str = config.GEMMA_TOKENIZER_REPO):
        from transformers import AutoTokenizer
        self.repo = repo
        self.tok = AutoTokenizer.from_pretrained(repo)
        self.name = f"{type(self.tok).__name__} ({repo})"

    @lru_cache(maxsize=4096)
    def count(self, text: str) -> int:
        return len(self.tok.encode(text, add_special_tokens=False))

    def count_chat(self, messages) -> int:
        """Local estimate of prompt tokens after applying the Gemma chat template."""
        ids = self.tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        if isinstance(ids, dict) or hasattr(ids, "keys"):
            ids = ids["input_ids"]
        return len(ids)


def tiktoken_gemma_check() -> str:
    """Document whether tiktoken can map the Gemma model (expected: no)."""
    try:
        import tiktoken
    except ImportError:
        return "tiktoken not installed"
    try:
        enc = tiktoken.encoding_for_model(config.OPENAI_MODEL)
        return f"tiktoken mapped {config.OPENAI_MODEL} to {enc.name} (unexpected)"
    except Exception as e:  # noqa: BLE001
        return f"tiktoken cannot map {config.OPENAI_MODEL}: {e}"

## 4. Embeddings and the shared FAISS index
Chunks are embedded with `all-MiniLM-L6-v2` (L2-normalised) and stored in an exact `IndexFlatIP` index, so inner product equals cosine similarity.
The index is built once, persisted in `index/`, and used by **both** pipelines. It is rebuilt automatically if the documents, embedding model or chunking settings change.

In [ ]:
"""Shared FAISS index (built once, used by both pipelines)."""
import hashlib
import json

import numpy as np



def _docs_fingerprint():
    h = hashlib.sha256()
    for p in sorted(config.DOCS_DIR.glob("*.md")):
        h.update(p.name.encode())
        h.update(p.read_bytes())
    return h.hexdigest()


def load_embedder():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)


def embed(embedder, texts):
    vecs = embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True,
                           show_progress_bar=False, batch_size=32)
    return np.asarray(vecs, dtype="float32")


def build_or_load_index(embedder, rebuild: bool = False):
    """Returns (faiss_index, chunks, manifest). Cosine similarity via inner product on L2-normalised vectors."""
    import faiss
    config.INDEX_DIR.mkdir(parents=True, exist_ok=True)
    idx_path = config.INDEX_DIR / "faiss.index"
    chunks_path = config.INDEX_DIR / "chunks.json"
    manifest_path = config.INDEX_DIR / "manifest.json"
    fp = _docs_fingerprint()
    if not rebuild and idx_path.exists() and chunks_path.exists() and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (manifest.get("docs_sha256") == fp and manifest.get("embedding_model") == config.EMBEDDING_MODEL
                and manifest.get("chunk_size") == config.CHUNK_SIZE
                and manifest.get("chunk_overlap") == config.CHUNK_OVERLAP):
            index = faiss.read_index(str(idx_path))
            chunks = [Chunk(**c) for c in json.loads(chunks_path.read_text(encoding="utf-8"))]
            manifest["loaded_from_disk"] = True
            return index, chunks, manifest
    chunks = load_and_chunk()
    vecs = embed(embedder, [c.text for c in chunks])
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    faiss.write_index(index, str(idx_path))
    chunks_path.write_text(json.dumps([c.to_dict() for c in chunks], indent=2, ensure_ascii=False), encoding="utf-8")
    manifest = {
        "docs_sha256": fp, "embedding_model": config.EMBEDDING_MODEL, "embedding_dim": int(vecs.shape[1]),
        "faiss_index_type": "IndexFlatIP (exact search, cosine via L2-normalised vectors)",
        "num_chunks": len(chunks), "chunk_size": config.CHUNK_SIZE, "chunk_overlap": config.CHUNK_OVERLAP,
        "faiss_version": faiss.__version__,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    manifest["loaded_from_disk"] = False
    return index, chunks, manifest

## 5. Extractive contextual compression
This step is local and deterministic. It **selects original sentences only** and never summarises, paraphrases or generates text. Every kept sentence stays under its source chunk ID, and the full rules are in the docstring below.

In [ ]:
"""Deterministic LOCAL EXTRACTIVE compression.

Rules implemented (fixed before any evaluation run):
1. Split each selected chunk into segments: first on line breaks, then into
   sentences with a regex. Text is kept verbatim (no summarising/paraphrasing).
2. Remove duplicate segments created by the 80-char chunk overlap: a segment
   whose whitespace-normalised text is contained in another candidate segment
   is dropped (the longer, more complete one is kept).
3. Score every remaining segment against the question with the same
   cross-encoder used for reranking (logit score).
4. Select segments in descending score order: a segment is kept if its score
   >= COMPRESSION_SCORE_THRESHOLD, or if fewer than COMPRESSION_MIN_SEGMENTS
   have been kept so far; selection stops before the formatted context would
   exceed COMPRESSION_TOKEN_TARGET tokens.
5. Kept segments are re-ordered into original reading order (by reranked
   chunk rank, then position) and grouped under their source chunk ID.
Ties are broken by (chunk rank, position) so the output is deterministic.
"""
import re
from dataclasses import dataclass, asdict


_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(\[])")
_MARKER = re.compile(r"^(#+\s*)?\d+\.$|^#+$")


@dataclass
class Segment:
    segment_id: str       # e.g. "IT-C05#S03"
    chunk_id: str
    chunk_rank: int       # rank of the chunk after reranking (0 = best)
    position: int         # position of the segment inside its chunk
    text: str
    score: float = 0.0
    selected: bool = False


def split_segments(chunk_text: str):
    segs = []
    for line in chunk_text.split("\n"):
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in _SENT_SPLIT.split(line) if p.strip()]
        merged = []
        for p in parts:
            # re-attach list/heading markers such as "1." or "## 7." to the text that follows,
            # recovering the original substring of the line
            if merged and _MARKER.match(merged[-1]):
                start = line.find(merged[-1])
                end = line.find(p, start + len(merged[-1])) + len(p)
                merged[-1] = line[start:end]
            else:
                merged.append(p)
        segs.extend(merged)
    return segs


def _norm(s: str) -> str:
    return " ".join(s.split()).lower()


def format_compressed_context(segments):
    """Group selected segments under their chunk ID, in reading order."""
    blocks, current, lines = [], None, []
    for seg in segments:
        if seg.chunk_id != current:
            if current is not None:
                blocks.append(f"[{current}]\n" + "\n".join(lines))
            current, lines = seg.chunk_id, []
        lines.append(seg.text)
    if current is not None:
        blocks.append(f"[{current}]\n" + "\n".join(lines))
    return "\n\n".join(blocks)


def compress(question, ranked_chunks, cross_encoder, token_counter,
             token_target=config.COMPRESSION_TOKEN_TARGET,
             threshold=config.COMPRESSION_SCORE_THRESHOLD,
             min_segments=config.COMPRESSION_MIN_SEGMENTS):
    """ranked_chunks: list of Chunk in reranked order. Returns (context_str, all_segments, selected_segments, budget_hit)."""
    cands = []
    for rank, ch in enumerate(ranked_chunks):
        for pos, text in enumerate(split_segments(ch.text)):
            cands.append(Segment(f"{ch.chunk_id}#S{pos:02d}", ch.chunk_id, rank, pos, text))

    # de-duplicate overlap fragments (keep the longer / earlier-ranked one)
    normed = [_norm(c.text) for c in cands]
    keep = []
    for i, c in enumerate(cands):
        dup = False
        for j, other in enumerate(cands):
            if i == j:
                continue
            if normed[i] == normed[j]:
                if j < i:
                    dup = True
                    break
            elif normed[i] in normed[j]:
                dup = True
                break
        if not dup:
            keep.append(c)
    cands = keep

    if cands:
        scores = cross_encoder.predict([(question, c.text) for c in cands], show_progress_bar=False)
        for c, s in zip(cands, scores):
            c.score = float(s)

    order = sorted(cands, key=lambda c: (-c.score, c.chunk_rank, c.position))
    selected = []
    budget_hit = False
    for c in order:
        if c.score < threshold and len(selected) >= min_segments:
            break  # remaining segments all score lower
        trial = sorted(selected + [c], key=lambda x: (x.chunk_rank, x.position))
        if token_counter.count(format_compressed_context(trial)) > token_target:
            budget_hit = True
            break
        selected.append(c)
    selected = sorted(selected, key=lambda x: (x.chunk_rank, x.position))
    for c in selected:
        c.selected = True
    return format_compressed_context(selected), cands, selected, budget_hit


def segments_to_records(segments):
    return [asdict(s) for s in segments]

## 6. LM Studio client
The OpenAI Python SDK is used **only** as an OpenAI-compatible client for the local LM Studio server; no cloud API or API key is involved.
Streaming is used so that time-to-first-token (TTFT) can be measured. Only transient connection errors are retried, and every retry is recorded.

In [ ]:
"""LM Studio client (OpenAI-compatible API used ONLY as a local client)."""
import json
import time
import urllib.request

from openai import OpenAI, APIConnectionError, APITimeoutError, InternalServerError


TRANSIENT_ERRORS = (APIConnectionError, APITimeoutError, InternalServerError)


def get_client():
    return OpenAI(base_url=config.OPENAI_BASE_URL, api_key=config.OPENAI_API_KEY,
                  timeout=config.LLM_TIMEOUT_SECONDS, max_retries=0)


def list_models():
    with urllib.request.urlopen(config.OPENAI_BASE_URL.rstrip("/") + "/models", timeout=10) as r:
        return json.loads(r.read().decode())


def lmstudio_native_model_info(model_id):
    """LM Studio's REST API (/api/v0/models/<id>) may expose arch/quantization. Optional."""
    base = config.OPENAI_BASE_URL.rstrip("/")
    if base.endswith("/v1"):
        base = base[:-3]
    try:
        with urllib.request.urlopen(f"{base}/api/v0/models/{model_id}", timeout=10) as r:
            return json.loads(r.read().decode())
    except Exception as e:  # noqa: BLE001
        return {"unavailable": f"{type(e).__name__}: {e}"}


def build_messages(context: str, question: str):
    return [
        {"role": "system", "content": config.SYSTEM_PROMPT},
        {"role": "user", "content": config.USER_TEMPLATE.format(context=context, question=question)},
    ]


def _generate_once(client, messages):
    t0 = time.perf_counter()
    ttft = None
    usage = None
    finish_reason = None
    if config.LLM_STREAM:
        stream = client.chat.completions.create(
            model=config.OPENAI_MODEL, messages=messages, stream=True,
            stream_options={"include_usage": True}, **config.GENERATION_PARAMS)
        parts = []
        for ev in stream:
            if ev.choices:
                delta = ev.choices[0].delta
                if delta is not None and delta.content:
                    if ttft is None:
                        ttft = time.perf_counter() - t0
                    parts.append(delta.content)
                if ev.choices[0].finish_reason:
                    finish_reason = ev.choices[0].finish_reason
            if getattr(ev, "usage", None):
                usage = ev.usage
        text = "".join(parts)
    else:
        resp = client.chat.completions.create(model=config.OPENAI_MODEL, messages=messages,
                                              **config.GENERATION_PARAMS)
        text = resp.choices[0].message.content or ""
        usage = resp.usage
        finish_reason = resp.choices[0].finish_reason
    gen_time = time.perf_counter() - t0
    return {
        "answer": text,
        "generation_time_seconds": gen_time,
        "ttft_seconds": ttft,
        "server_prompt_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
        "server_completion_tokens": getattr(usage, "completion_tokens", None) if usage else None,
        "server_total_tokens": getattr(usage, "total_tokens", None) if usage else None,
        "finish_reason": finish_reason,
    }


def generate(client, messages):
    """Generate with retries ONLY for transient infrastructure errors. Returns dict incl. retry info."""
    retries, errors = 0, []
    while True:
        try:
            out = _generate_once(client, messages)
            out["retry_count"] = retries
            out["retry_errors"] = " | ".join(errors)
            return out
        except TRANSIENT_ERRORS as e:
            errors.append(f"{type(e).__name__}: {e}")
            if retries >= config.MAX_TRANSIENT_RETRIES:
                raise RuntimeError(f"LLM failed after {retries} retries: {' | '.join(errors)}") from e
            retries += 1
            time.sleep(5 * retries)

## 7. The two RAG pipelines
The latency definitions are given in the docstring. Token counting that is done only for measurement happens **after** the timers stop, so it does not inflate latency.

In [ ]:
"""Baseline and token-efficient RAG pipelines.

Latency definitions (all wall-clock, time.perf_counter(), same machine):
  retrieval_time_seconds    query embedding + FAISS search + fetching chunk text
  reranking_time_seconds    cross-encoder scoring of the 10 candidates + sort (optimized only; 0 for baseline)
  compression_time_seconds  sentence split + dedup + cross-encoder sentence scoring + selection (optimized only)
  generation_time_seconds   LM Studio chat-completion call (request sent -> last streamed token received)
  total_latency_seconds     whole pipeline: from receiving the question to having the final answer,
                            including prompt construction and any transient-error retries.
Local token counting done only for measurement is excluded from the timed stages
(it is performed after the stage timers stop) so that it does not inflate latency.
"""
import json
import re
import time

import numpy as np


CITATION_RE = re.compile(r"\[([^\[\]]+)\]")
CHUNK_ID_RE = re.compile(r"\b([A-Z]+-C\d{2})\b")


def format_full_context(chunks):
    return "\n\n".join(f"[{c.chunk_id}]\n{c.text}" for c in chunks)


def extract_cited_ids(answer: str):
    ids = []
    for grp in CITATION_RE.findall(answer or ""):
        for cid in CHUNK_ID_RE.findall(grp):
            if cid not in ids:
                ids.append(cid)
    return ids


class RAGSystem:
    def __init__(self, embedder, index, chunks, cross_encoder, token_counter, llm_client):
        self.embedder = embedder
        self.index = index
        self.chunks = chunks
        self.ce = cross_encoder
        self.tc = token_counter
        self.client = llm_client

    # ---------- shared ----------
    def _search(self, question, k):
        qv = embed(self.embedder, [question])
        scores, ids = self.index.search(qv, k)
        return [(self.chunks[i], float(s)) for i, s in zip(ids[0], scores[0]) if i != -1]

    def _finish(self, rec, question, context, t_start):
        messages = build_messages(context, question)
        gen = generate(self.client, messages)
        rec["total_latency_seconds"] = time.perf_counter() - t_start
        rec.update(gen)
        # measurement-only work below (outside the timed window)
        rec["final_context_tokens"] = self.tc.count(context)
        rec["local_prompt_tokens_estimate"] = self.tc.count_chat(messages)
        rec["local_output_tokens_estimate"] = self.tc.count(gen["answer"])
        if gen["server_prompt_tokens"] is not None:
            rec["input_prompt_tokens"] = gen["server_prompt_tokens"]
            rec["output_tokens"] = gen["server_completion_tokens"]
            rec["token_count_source"] = "lm_studio_usage"
        else:
            rec["input_prompt_tokens"] = rec["local_prompt_tokens_estimate"]
            rec["output_tokens"] = rec["local_output_tokens_estimate"]
            rec["token_count_source"] = "local_gemma_tokenizer (server usage missing)"
        rec["total_tokens"] = rec["input_prompt_tokens"] + rec["output_tokens"]
        gt = rec["generation_time_seconds"]
        rec["tokens_per_second"] = rec["output_tokens"] / gt if gt and gt > 0 else None
        rec["final_context"] = context
        rec["system_prompt"] = messages[0]["content"]
        rec["user_message"] = messages[1]["content"]
        rec["generated_answer"] = gen["answer"]
        cited = extract_cited_ids(gen["answer"])
        rec["source_chunk_ids"] = json.dumps(cited)
        rec["cited_ids_subset_of_final_chunks"] = all(c in json.loads(rec["final_chunks"]) for c in cited)
        rec["refusal_exact_match"] = config.REFUSAL_TEXT in (gen["answer"] or "")
        return rec

    # ---------- A. baseline ----------
    def run_baseline(self, question):
        t_start = time.perf_counter()
        t0 = time.perf_counter()
        hits = self._search(question, config.BASELINE_TOP_K)
        retrieved = [c for c, _ in hits]
        context = format_full_context(retrieved)
        retrieval_time = time.perf_counter() - t0
        rec = {
            "approach_name": "baseline",
            "retrieved_chunks": json.dumps([c.chunk_id for c in retrieved]),
            "retrieval_scores": json.dumps([round(s, 6) for _, s in hits]),
            "reranker_scores": json.dumps(None),
            "final_chunks": json.dumps([c.chunk_id for c in retrieved]),
            "selected_segments": json.dumps(None),
            "retrieval_time_seconds": retrieval_time,
            "reranking_time_seconds": 0.0,
            "compression_time_seconds": 0.0,
            "compression_budget_hit": None,
        }
        rec = self._finish(rec, question, context, t_start)
        rec["raw_retrieved_context_tokens"] = self.tc.count(format_full_context(retrieved))
        rec["pre_compression_context_tokens"] = rec["raw_retrieved_context_tokens"]
        return rec

    # ---------- B. token-efficient ----------
    def run_optimized(self, question):
        t_start = time.perf_counter()
        t0 = time.perf_counter()
        hits = self._search(question, config.OPTIMIZED_INITIAL_K)
        initial = [c for c, _ in hits]
        retrieval_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        ce_scores = self.ce.predict([(question, c.text) for c in initial], show_progress_bar=False)
        order = sorted(range(len(initial)), key=lambda i: (-float(ce_scores[i]), i))
        top = [initial[i] for i in order[:config.OPTIMIZED_FINAL_K]]
        rerank_time = time.perf_counter() - t1

        t2 = time.perf_counter()
        context, all_segs, selected, budget_hit = compress(question, top, self.ce, self.tc)
        compression_time = time.perf_counter() - t2

        rec = {
            "approach_name": "optimized",
            "retrieved_chunks": json.dumps([c.chunk_id for c in initial]),
            "retrieval_scores": json.dumps([round(s, 6) for _, s in hits]),
            "reranker_scores": json.dumps({initial[i].chunk_id: round(float(ce_scores[i]), 6) for i in order}),
            "final_chunks": json.dumps([c.chunk_id for c in top]),
            "selected_segments": json.dumps([
                {"segment_id": s.segment_id, "score": round(s.score, 6)} for s in selected]),
            "all_segments": json.dumps(segments_to_records(all_segs), ensure_ascii=False),
            "retrieval_time_seconds": retrieval_time,
            "reranking_time_seconds": rerank_time,
            "compression_time_seconds": compression_time,
            "compression_budget_hit": budget_hit,
        }
        rec = self._finish(rec, question, context, t_start)
        rec["raw_retrieved_context_tokens"] = self.tc.count(format_full_context(initial))
        rec["pre_compression_context_tokens"] = self.tc.count(format_full_context(top))
        return rec

## 8. Preflight
Runs before any measurement. It checks:
- the required packages are installed;
- LM Studio is reachable and reports the configured model ID;
- tiktoken cannot map Gemma (this is recorded);
- the tokenizer, embedder and reranker load, and the FAISS index is rebuilt;
- a **live token cross-check**: the local chat-template token count equals LM Studio's `prompt_tokens`.

In [ ]:
import importlib
import json


def load_components(rebuild_index=False):
    """Load the tokenizer, embedder, reranker and the shared FAISS index (used by BOTH pipelines)."""
    import torch
    torch.manual_seed(0)
    from sentence_transformers import CrossEncoder
    tc = GemmaTokenCounter()
    embedder = load_embedder()
    ce = CrossEncoder(config.RERANKER_MODEL, device=config.DEVICE)
    index, chunks, manifest = build_or_load_index(embedder, rebuild=rebuild_index)
    return {"tc": tc, "embedder": embedder, "ce": ce, "index": index, "chunks": chunks, "manifest": manifest}


def preflight():
    """Checks packages, LM Studio, model ID, tokenizer (incl. a live token-count cross-check),
    encoders, and (re)builds the index. Saves results/preflight.json. Raises on any hard failure."""
    report = {"packages": {}}
    for p in ["openai", "sentence_transformers", "faiss", "transformers", "torch", "pandas",
              "matplotlib", "scipy", "psutil", "dotenv", "tiktoken"]:
        try:
            m = importlib.import_module(p)
            report["packages"][p] = getattr(m, "__version__", "ok")
        except Exception as e:  # noqa: BLE001
            report["packages"][p] = f"MISSING ({e})"
    missing = [p for p, v in report["packages"].items() if str(v).startswith("MISSING")]
    assert not missing, f"Missing packages: {missing}"

    ids = [m["id"] for m in list_models()["data"]]
    report["lm_studio_models"] = ids
    assert config.OPENAI_MODEL in ids, f"Model {config.OPENAI_MODEL!r} not loaded in LM Studio (got {ids})"
    report["lm_studio_native_model_info"] = lmstudio_native_model_info(config.OPENAI_MODEL)
    report["tiktoken_check"] = tiktoken_gemma_check()

    comps = load_components(rebuild_index=True)
    report["tokenizer"] = comps["tc"].name
    report["index"] = comps["manifest"]

    # Live cross-check: local Gemma chat-template count vs LM Studio's own prompt_tokens (max_tokens=1).
    msgs = build_messages("[TEST-C00]\nThis is a tokenizer check.", "Is this a tokenizer check?")
    resp = get_client().chat.completions.create(model=config.OPENAI_MODEL, messages=msgs,
                                                temperature=0.0, max_tokens=1)
    report["token_crosscheck"] = {"local_chat_template_tokens": comps["tc"].count_chat(msgs),
                                  "lm_studio_prompt_tokens": resp.usage.prompt_tokens if resp.usage else None}
    tcx = report["token_crosscheck"]
    tcx["match"] = tcx["local_chat_template_tokens"] == tcx["lm_studio_prompt_tokens"]

    config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    (config.RESULTS_DIR / "preflight.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    print(json.dumps({k: report[k] for k in ["lm_studio_models", "tokenizer", "tiktoken_check", "token_crosscheck"]},
                     indent=2, default=str))
    print("index:", comps["manifest"]["num_chunks"], "chunks |", comps["manifest"]["faiss_index_type"])
    print("PREFLIGHT OK" if tcx["match"] else "PREFLIGHT OK (WARNING: local vs server token counts differ)")
    return comps

## 9. Integration tests (real models, no LLM generation)
These tests use the real embedder, index, reranker and tokenizer:
- embeddings are deterministic and normalised;
- every chunk retrieves itself at rank 1 (FAISS self-retrieval);
- the reranker is deterministic;
- the baseline top-5 equals the first 5 of the optimized top-10 (same index and same query embedding);
- compression is verbatim, deterministic and within budget;
- the tokenizer passes a sanity check.

In [ ]:
import numpy as np


def run_integration_tests(comps):
    """Tests with the REAL embedder, index, reranker and tokenizer (no LLM generation).
    Saved to results/integration_tests.json. Raises AssertionError on failure."""
    tc, emb, ce, index, chunks = comps["tc"], comps["embedder"], comps["ce"], comps["index"], comps["chunks"]
    out = {}

    # 1. Embeddings are deterministic and L2-normalised
    v1, v2 = embed(emb, [chunks[0].text]), embed(emb, [chunks[0].text])
    assert np.allclose(v1, v2), "embedder not deterministic"
    assert abs(float(np.linalg.norm(v1)) - 1.0) < 1e-4, "embeddings not normalised"
    out["embedding_deterministic_and_normalised"] = True

    # 2. FAISS self-retrieval: every chunk's own text must retrieve that chunk at rank 1
    vecs = embed(emb, [c.text for c in chunks])
    _, ids = index.search(vecs, 1)
    hits = sum(int(ids[i][0] == i) for i in range(len(chunks)))
    assert hits == len(chunks), f"self-retrieval {hits}/{len(chunks)}"
    out["faiss_self_retrieval"] = f"{hits}/{len(chunks)}"

    # 3. Index size matches chunk list; both pipelines search the same index object
    assert index.ntotal == len(chunks)
    out["index_ntotal"] = int(index.ntotal)

    # 4. Reranker is deterministic
    pairs = [("How many days of annual leave?", c.text) for c in chunks[:10]]
    assert np.allclose(ce.predict(pairs, show_progress_bar=False), ce.predict(pairs, show_progress_bar=False))
    out["reranker_deterministic"] = True

    # 5. Retrieval stages without the LLM: sizes, overlap and verbatim, deterministic compression
    rag = RAGSystem(emb, index, chunks, ce, tc, llm_client=None)
    q = "What steps should store staff follow when a POS terminal fails?"
    base_ids = [c.chunk_id for c, _ in rag._search(q, config.BASELINE_TOP_K)]
    init = [c for c, _ in rag._search(q, config.OPTIMIZED_INITIAL_K)]
    assert len(base_ids) == 5 and len(init) == 10 and base_ids == [c.chunk_id for c in init[:5]], \
        "top-5 must be the first 5 of top-10 (same index, same query embedding)"
    scores = ce.predict([(q, c.text) for c in init], show_progress_bar=False)
    top = [init[i] for i in sorted(range(10), key=lambda i: (-float(scores[i]), i))[:3]]
    ctx1, _, sel1, _ = compress(q, top, ce, tc)
    ctx2, _, sel2, _ = compress(q, top, ce, tc)
    assert ctx1 == ctx2, "compression not deterministic"
    assert all(s.text in top[s.chunk_rank].text for s in sel1), "compressed text is not verbatim"
    assert tc.count(ctx1) <= config.COMPRESSION_TOKEN_TARGET
    out["retrieval_and_compression_checks"] = {"baseline_top5": base_ids, "optimized_top3": [c.chunk_id for c in top],
                                               "compressed_tokens": tc.count(ctx1)}

    # 6. Tokenizer sanity: Gemma tokenizer differs from a character count and is stable
    t = "Employees accrue 1.75 days of paid annual leave."
    assert 0 < tc.count(t) < len(t) and tc.count(t) == tc.count(t)
    out["tokenizer_sanity"] = {"text": t, "tokens": tc.count(t), "chars": len(t)}

    (config.RESULTS_DIR / "integration_tests.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
    print(json.dumps(out, indent=2))
    print("ALL INTEGRATION TESTS PASSED")
    return out

## 10. Experiment runner
- **Warm-up:** one question outside the evaluation set is run through both pipelines. It is logged in `warmup_log.json` and never mixed with results.
- **Repetitions:** `N_RUNS` complete passes. The pipeline order alternates by question and flips on every pass, which balances order, caching and thermal effects.
- **Traceability:** each row is written to `experiment_results.csv` and `experiment_details.jsonl` as soon as it finishes. Progress also goes to `run_log.txt`.
- **Failures** are recorded with `error_message` and the post-failure `/models` status; they are never skipped. Earlier results are archived, not overwritten.

In [ ]:
import csv
import datetime as dt
import importlib.metadata as im
import json
import platform
import shutil
import subprocess
import time
import traceback

SMOKE_QUESTION_IDS = ["Q01", "Q21"]  # one answerable + one unanswerable
WARMUP_QUESTION = "What is the purpose of the Northstar policies?"  # not part of the evaluation set

CSV_COLUMNS = [
    "run_id", "question_id", "question", "answerable", "category", "approach_name", "model_name",
    "embedding_model", "reranker", "retrieved_chunks", "final_chunks",
    "raw_retrieved_context_tokens", "final_context_tokens", "input_prompt_tokens", "output_tokens",
    "total_tokens", "retrieval_time_seconds", "reranking_time_seconds", "compression_time_seconds",
    "generation_time_seconds", "total_latency_seconds", "tokens_per_second", "source_chunk_ids",
    "generated_answer", "error_message",
    # additional traceability columns
    "pre_compression_context_tokens", "local_prompt_tokens_estimate", "local_output_tokens_estimate",
    "token_count_source", "ttft_seconds", "finish_reason", "retrieval_scores", "reranker_scores",
    "selected_segments", "compression_budget_hit", "cited_ids_subset_of_final_chunks",
    "refusal_exact_match", "retry_count", "retry_errors", "run_order_position", "attempt",
    "run_started_at_utc", "final_context",
]


def pkg_version(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None


def hardware_info():
    info = {"platform": platform.platform(), "system": platform.system(), "release": platform.release(),
            "machine": platform.machine(), "processor": platform.processor() or None}
    try:
        import psutil
        info["cpu_physical_cores"] = psutil.cpu_count(logical=False)
        info["cpu_logical_cores"] = psutil.cpu_count(logical=True)
        info["ram_total_gb"] = round(psutil.virtual_memory().total / 1024 ** 3, 2)
    except ImportError:
        info["psutil"] = "not installed; core/RAM info unavailable"
    if platform.system() == "Windows":
        try:
            out = subprocess.run(["powershell", "-NoProfile", "-Command",
                                  "(Get-CimInstance Win32_Processor).Name; (Get-CimInstance Win32_VideoController).Name"],
                                 capture_output=True, text=True, timeout=30)
            lines = [l.strip() for l in out.stdout.splitlines() if l.strip()]
            if lines:
                info["cpu_name"] = lines[0]
                info["gpu_names"] = lines[1:]
        except Exception as e:  # noqa: BLE001
            info["cpu_gpu_query_error"] = str(e)
    if platform.system() == "Linux":
        try:
            names = [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")]
            if names:
                info["cpu_name"] = names[0]
        except OSError:
            pass
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=30)
            info["nvidia_smi"] = out.stdout.strip()
        except Exception as e:  # noqa: BLE001
            info["nvidia_smi_error"] = str(e)
    return info


def build_metadata(tc, manifest, models_resp, native_info, warmup, mode, question_ids, n_runs):
    import torch
    return {
        "experiment_title": "Token-Efficient RAG: Reducing Context Size and Latency Through Reranking and Contextual Compression",
        "research_question": ("To what extent can reranking and extractive contextual compression reduce "
                              "context-token usage and response latency in a local RAG system while maintaining answer quality?"),
        "run_mode": mode,
        "executed_from": "Token_Efficient_RAG.ipynb",
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python_version": sys.version,
        "hardware_os": hardware_info(),
        "llm_runtime": getattr(config, "RUNTIME_INFO", None),
        "llm": {
            "provider": config.LLM_PROVIDER, "base_url": config.OPENAI_BASE_URL,
            "api_key_note": "placeholder 'lm-studio' (local server, no secret)",
            "model_id": config.OPENAI_MODEL, "model_repository_reported_by_user": config.MODEL_REPOSITORY,
            "quantization_reported_by_user": config.MODEL_QUANTIZATION,
            "lm_studio_v1_models_response": models_resp, "lm_studio_native_model_info": native_info,
            "generation_params": config.GENERATION_PARAMS, "streaming": config.LLM_STREAM,
            "timeout_seconds": config.LLM_TIMEOUT_SECONDS, "max_transient_retries": config.MAX_TRANSIENT_RETRIES,
        },
        "prompt": {"system_prompt": config.SYSTEM_PROMPT, "user_template": config.USER_TEMPLATE,
                   "identical_for_both_pipelines": True},
        "data": {"documents_dir": "data/documents", "questions_file": "data/eval_questions.json",
                 "question_ids_run": question_ids, "doc_id_map": config.DOC_ID_MAP},
        "index": manifest,
        "chunking": {"chunk_size_chars": config.CHUNK_SIZE, "chunk_overlap_chars": config.CHUNK_OVERLAP,
                     "method": "deterministic character windows ending at last whitespace; next window starts overlap chars earlier at a word boundary"},
        "embedding_model": config.EMBEDDING_MODEL,
        "reranker_model": config.RERANKER_MODEL,
        "local_encoder_device": config.DEVICE,
        "baseline": {"top_k": config.BASELINE_TOP_K, "rerank": False, "compression": False},
        "optimized": {
            "initial_top_k": config.OPTIMIZED_INITIAL_K, "final_top_k_after_rerank": config.OPTIMIZED_FINAL_K,
            "compression_method": ("extractive: line+regex sentence split of the top-3 reranked chunks, overlap de-duplication, "
                                   "cross-encoder (same reranker model) relevance score per segment, keep segments with logit >= "
                                   f"{config.COMPRESSION_SCORE_THRESHOLD} (minimum {config.COMPRESSION_MIN_SEGMENTS} segments), "
                                   f"stop before formatted context exceeds {config.COMPRESSION_TOKEN_TARGET} tokens; verbatim text, reading order preserved"),
            "compression_token_target": config.COMPRESSION_TOKEN_TARGET,
            "compression_score_threshold_logit": config.COMPRESSION_SCORE_THRESHOLD,
            "compression_min_segments": config.COMPRESSION_MIN_SEGMENTS,
            "compression_parameters_chosen": "a priori, before any evaluation run; not tuned on evaluation questions",
        },
        "token_counting": {
            "local_tokenizer": tc.name,
            "tiktoken_check": tiktoken_gemma_check(),
            "raw_retrieved_context_tokens": "local Gemma tokenizer on formatted retrieved chunks (baseline: top-5; optimized: initial top-10)",
            "final_context_tokens": "local Gemma tokenizer on the exact context string sent to the LLM",
            "input_prompt_tokens": "LM Studio usage.prompt_tokens (loaded GGUF tokenizer incl. chat template); local estimate kept separately",
            "output_tokens": "LM Studio usage.completion_tokens; local estimate kept separately",
            "total_tokens": "input_prompt_tokens + output_tokens",
        },
        "latency_definitions": {
            "timer": "time.perf_counter()",
            "retrieval_time_seconds": "query embedding + FAISS search + chunk text lookup",
            "reranking_time_seconds": "cross-encoder scoring of initial candidates + sort (0 for baseline)",
            "compression_time_seconds": "segment split + dedup + cross-encoder segment scoring + token-budgeted selection (0 for baseline)",
            "generation_time_seconds": "LM Studio streamed chat completion, request to last token",
            "ttft_seconds": "time to first streamed content token (reflects prompt processing)",
            "total_latency_seconds": "question received -> final answer, incl. prompt construction and any retries",
            "excluded": "model loading, index building, measurement-only token counting",
        },
        "n_runs": n_runs,
        "run_order": ("N_RUNS complete passes; within a pass questions run in file order and both pipelines run "
                      "back-to-back for each question; pipeline order = baseline first when (question_index + run_id - 1) "
                      "is even, else optimized first, so the order flips on every pass (balances cache/thermal/order effects)"),
        "warmup": warmup,
        "software_versions": {p: pkg_version(p) for p in [
            "openai", "sentence-transformers", "transformers", "tokenizers", "torch", "faiss-cpu",
            "numpy", "pandas", "matplotlib", "tiktoken", "psutil", "scipy"]},
        "torch_threads": torch.get_num_threads(),
        "manual_evaluation": "quality/groundedness/refusal scores are NOT assigned automatically; see manual_evaluation_template.csv",
    }


def load_questions(ids=None):
    qs = json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))
    if ids:
        qs = [q for q in qs if q["question_id"] in ids]
    return qs


def _log(msg, log_path=None):
    print(msg, flush=True)
    if log_path is not None:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(msg + "\n")


def run_experiment(smoke=False, resume=False, n_runs=None, comps=None):
    """smoke=True  -> 1 pass over SMOKE_QUESTION_IDS, written to results/smoke_test/
    resume=True -> continue a full run: successful (run_id, question, pipeline) rows are kept, others re-attempted.
    Otherwise   -> N_RUNS full passes; previous top-level result files are archived (never deleted)."""
    n_runs = 1 if smoke else (n_runs or config.N_RUNS)
    out_dir = config.RESULTS_DIR / "smoke_test" if smoke else config.RESULTS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path, jsonl_path = out_dir / "experiment_results.csv", out_dir / "experiment_details.jsonl"
    log_path = out_dir / "run_log.txt"

    # ---- preflight: the configured model must be the one LM Studio reports ----
    models_resp = list_models()
    ids = [m.get("id") for m in models_resp.get("data", [])]
    if config.OPENAI_MODEL not in ids:
        raise RuntimeError(f"ABORT: model '{config.OPENAI_MODEL}' not reported by LM Studio (got {ids}).")
    native_info = lmstudio_native_model_info(config.OPENAI_MODEL)

    comps = comps or load_components()
    rag = RAGSystem(comps["embedder"], comps["index"], comps["chunks"], comps["ce"], comps["tc"], get_client())
    questions = load_questions(SMOKE_QUESTION_IDS if smoke else None)
    if not smoke and len(questions) != 24:
        raise RuntimeError(f"ABORT: expected 24 questions, found {len(questions)}")

    # ---- existing results: resume, or archive (never overwrite silently) ----
    done, attempts = set(), {}
    if csv_path.exists() and resume:
        with open(csv_path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                key = (int(r.get("run_id") or 1), r["question_id"], r["approach_name"])
                attempts[key] = max(attempts.get(key, 0), int(r.get("attempt") or 1))
                if not r["error_message"]:
                    done.add(key)
        _log(f"[resume] {len(done)} successful runs found; they will not be rerun", log_path)
    elif csv_path.exists() or log_path.exists():
        arch = out_dir / "archive" / dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        arch.mkdir(parents=True, exist_ok=True)
        for p in [csv_path, jsonl_path, log_path, out_dir / "experiment_metadata.json", out_dir / "warmup_log.json"]:
            if p.exists():
                shutil.move(str(p), str(arch / p.name))
        _log(f"[archive] previous results moved to {arch}", log_path)
    _log(f"[start] {dt.datetime.now().isoformat(timespec='seconds')} | runs={n_runs} | questions={len(questions)} | model={config.OPENAI_MODEL}", log_path)

    # ---- warm-up (recorded separately, never mixed with results) ----
    warmup = {"question": WARMUP_QUESTION, "runs": []}
    if not (resume and done):
        for fn, name in [(rag.run_baseline, "baseline"), (rag.run_optimized, "optimized")]:
            try:
                r = fn(WARMUP_QUESTION)
                warmup["runs"].append({"approach": name, "total_latency_seconds": r["total_latency_seconds"],
                                       "generation_time_seconds": r["generation_time_seconds"],
                                       "answer": r["generated_answer"]})
                _log(f"[warmup] {name}: {r['total_latency_seconds']:.2f}s", log_path)
            except Exception as e:  # noqa: BLE001
                warmup["runs"].append({"approach": name, "error": f"{type(e).__name__}: {e}"})
                _log(f"[warmup] {name} FAILED: {e}", log_path)
        (out_dir / "warmup_log.json").write_text(json.dumps(warmup, indent=2), encoding="utf-8")

    meta = build_metadata(comps["tc"], comps["manifest"], models_resp, native_info, warmup,
                          "smoke_test" if smoke else "full", [q["question_id"] for q in questions], n_runs)
    meta_path = out_dir / "experiment_metadata.json"
    if resume and meta_path.exists():
        old = json.loads(meta_path.read_text(encoding="utf-8"))
        old.setdefault("resume_sessions", []).append({"timestamp_utc": meta["timestamp_utc"],
                                                      "lm_studio_v1_models_response": models_resp})
        meta = old
    meta_path.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")

    # ---- main loop: every row is written immediately; failures are recorded, never skipped ----
    new_file = not csv_path.exists()
    fcsv = open(csv_path, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(fcsv, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    if new_file:
        writer.writeheader()
    fjs = open(jsonl_path, "a", encoding="utf-8")
    n_ok = n_fail = 0
    t_all = time.perf_counter()
    try:
        for run_id in range(1, n_runs + 1):
            for qi, q in enumerate(questions):
                order = [("baseline", rag.run_baseline), ("optimized", rag.run_optimized)]
                if (qi + run_id - 1) % 2 == 1:
                    order.reverse()
                for pos, (name, fn) in enumerate(order):
                    key = (run_id, q["question_id"], name)
                    if key in done:
                        continue
                    base = {"run_id": run_id, "question_id": q["question_id"], "question": q["question"],
                            "answerable": q["answerable"], "category": q["category"], "approach_name": name,
                            "model_name": config.OPENAI_MODEL, "embedding_model": config.EMBEDDING_MODEL,
                            "reranker": config.RERANKER_MODEL if name == "optimized" else "none",
                            "run_order_position": pos, "attempt": attempts.get(key, 0) + 1,
                            "run_started_at_utc": dt.datetime.now(dt.timezone.utc).isoformat()}
                    try:
                        rec = fn(q["question"])
                        rec.update(base)
                        rec["error_message"] = ""
                        n_ok += 1
                        _log(f"[run {run_id}/{n_runs}] {q['question_id']}/{name:9s} ok  ctx={rec['final_context_tokens']:4d} "
                             f"prompt={rec['input_prompt_tokens']:4d} out={rec['output_tokens']:4d} "
                             f"lat={rec['total_latency_seconds']:6.2f}s  ans={rec['generated_answer'][:70]!r}", log_path)
                    except Exception as e:  # noqa: BLE001
                        rec = dict(base)
                        rec["error_message"] = f"{type(e).__name__}: {e}"
                        rec["traceback"] = traceback.format_exc()
                        n_fail += 1
                        try:
                            rec["error_message"] += f" | post-failure /models: {[m.get('id') for m in list_models().get('data', [])]}"
                        except Exception as e2:  # noqa: BLE001
                            rec["error_message"] += f" | post-failure /models unreachable: {e2}"
                        _log(f"[run {run_id}/{n_runs}] {q['question_id']}/{name} FAILED: {rec['error_message']}", log_path)
                    writer.writerow(rec)
                    fcsv.flush()
                    fjs.write(json.dumps(rec, default=str, ensure_ascii=False) + "\n")
                    fjs.flush()
    finally:
        fcsv.close()
        fjs.close()
    _log(f"[done] successful runs: {n_ok}; failed runs: {n_fail}; wall time {(time.perf_counter() - t_all) / 60:.1f} min", log_path)
    _log(f"[done] results: {csv_path}", log_path)
    return {"ok": n_ok, "failed": n_fail}

## 11. Smoke-test gate
This gate checks the mechanics only: rows, errors, token and latency fields, server usage, answers, citation validity, and the number of chunks per pipeline. It does **not** judge answer quality.
The full experiment starts only if this gate passes.

In [ ]:
def smoke_gate():
    """Pipeline-mechanics gate on results/smoke_test/. Returns True only if every check passes.
    Answer quality is NOT judged here."""
    p = config.RESULTS_DIR / "smoke_test" / "experiment_results.csv"
    rows = list(csv.DictReader(open(p, encoding="utf-8")))
    fails = []
    expected = {(q, a) for q in SMOKE_QUESTION_IDS for a in ["baseline", "optimized"]}
    if len(rows) != len(expected):
        fails.append(f"expected {len(expected)} rows, got {len(rows)}")
    if {(r["question_id"], r["approach_name"]) for r in rows} != expected:
        fails.append("unexpected question/approach set")
    for r in rows:
        tag = f"{r['question_id']}/{r['approach_name']}"
        if r["error_message"]:
            fails.append(f"{tag}: error {r['error_message']}")
            continue
        for col in ["raw_retrieved_context_tokens", "final_context_tokens", "input_prompt_tokens", "output_tokens",
                    "total_tokens", "retrieval_time_seconds", "generation_time_seconds", "total_latency_seconds"]:
            try:
                if not float(r[col]) > 0:
                    fails.append(f"{tag}: {col} not > 0 ({r[col]})")
            except ValueError:
                fails.append(f"{tag}: {col} missing")
        if r["token_count_source"] != "lm_studio_usage":
            fails.append(f"{tag}: server usage missing (source={r['token_count_source']})")
        if not r["generated_answer"].strip():
            fails.append(f"{tag}: empty answer")
        if r["cited_ids_subset_of_final_chunks"] != "True":
            fails.append(f"{tag}: cited ids not in final context: {r['source_chunk_ids']}")
        n_final = len(json.loads(r["final_chunks"]))
        if n_final != (config.BASELINE_TOP_K if r["approach_name"] == "baseline" else config.OPTIMIZED_FINAL_K):
            fails.append(f"{tag}: wrong number of final chunks ({n_final})")
        print(f"{tag}: ctx={r['final_context_tokens']} prompt={r['input_prompt_tokens']} "
              f"(local est {r['local_prompt_tokens_estimate']}) out={r['output_tokens']} "
              f"lat={float(r['total_latency_seconds']):.2f}s cited={r['source_chunk_ids']}\n   answer: {r['generated_answer']!r}")
    if fails:
        print("SMOKE GATE FAILED:")
        for f_ in fails:
            print("  -", f_)
        return False
    print("SMOKE GATE PASSED")
    return True

## 12. Analysis: metrics, automated checks, reproducibility and charts
- **Unit of analysis:** the question. Each metric is averaged over runs per question and pipeline, then summarised over the 24 questions.
- **Percentage reduction:** `(baseline − optimized) / baseline × 100`. A negative value is reported as an increase, and no value is given when the baseline is 0.
- **Per-run summaries and reproducibility:** checks whether answers, contexts and token counts are identical across runs, and measures latency variation.
- **Automated checks:** exact refusal, citation validity and truncation. These are **not quality scores**.
- **Manual template:** the manual-evaluation template leaves every score blank, and existing manual scores are never overwritten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

APPROACHES = ["baseline", "optimized"]
LABELS = {"baseline": "Baseline RAG (top-5, full chunks)",
          "optimized": "Token-efficient RAG (top-10 → rerank → top-3 → compress)"}
SHORT = {"baseline": "Baseline", "optimized": "Token-efficient"}
COLORS = {"baseline": "#2a78d6", "optimized": "#eb6834"}          # validated categorical slots 1-2
STAGE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

METRICS = [
    ("raw_retrieved_context_tokens", "tokens"), ("pre_compression_context_tokens", "tokens"),
    ("final_context_tokens", "tokens"), ("input_prompt_tokens", "tokens"), ("output_tokens", "tokens"),
    ("total_tokens", "tokens"), ("retrieval_time_seconds", "s"), ("reranking_time_seconds", "s"),
    ("compression_time_seconds", "s"), ("ttft_seconds", "s"), ("generation_time_seconds", "s"),
    ("total_latency_seconds", "s"), ("tokens_per_second", "tokens/s"),
    ("derived_decode_time_seconds", "s"), ("derived_decode_tokens_per_second", "tokens/s"),
    ("derived_prefill_tokens_per_second", "tokens/s"),
]
# Derived here from recorded columns (not stored in experiment_results.csv):
#   derived_decode_time_seconds       = generation_time_seconds - ttft_seconds
#   derived_decode_tokens_per_second  = (output_tokens - 1) / derived_decode_time_seconds
#   derived_prefill_tokens_per_second = input_prompt_tokens / ttft_seconds (TTFT also includes HTTP/first-token overhead)
STRUCTURAL_ZERO_BASELINE = {"reranking_time_seconds", "compression_time_seconds"}
LOWER_IS_BETTER = {m for m, _ in METRICS if not m.endswith("tokens_per_second")}
SCORE_COLUMNS = ["baseline_quality", "optimized_quality", "baseline_groundedness", "optimized_groundedness",
                 "baseline_correct_refusal", "optimized_correct_refusal"]


def pct_reduction(b, o):
    """((baseline - optimized) / baseline) * 100; None when undefined (baseline 0 or missing)."""
    if b is None or o is None or not np.isfinite(b) or not np.isfinite(o) or b == 0:
        return None
    return (b - o) / b * 100.0


def describe(p):
    if p is None:
        return "unavailable (baseline is 0 or missing)"
    if abs(p) < 1e-9:
        return "no change"
    return "reduction" if p > 0 else "increase (negative reduction)"


def style_axes(ax, xgrid=False):
    ax.set_facecolor(SURFACE)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    if xgrid:
        ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def load(results_dir):
    df = pd.read_csv(Path(results_dir) / "experiment_results.csv", keep_default_na=False, na_values=[""])
    df["error_message"] = df["error_message"].fillna("")
    if "run_id" not in df.columns:
        df["run_id"] = 1
    df["run_id"] = df["run_id"].fillna(1).astype(int)
    df["answerable"] = df["answerable"].astype(str).str.lower() == "true"
    df["derived_decode_time_seconds"] = df["generation_time_seconds"] - df["ttft_seconds"]
    df["derived_decode_tokens_per_second"] = (df["output_tokens"] - 1) / df["derived_decode_time_seconds"]
    df["derived_prefill_tokens_per_second"] = df["input_prompt_tokens"] / df["ttft_seconds"]
    return df


def validate(df, qs):
    """Data-integrity checks. Returns a list of issues (empty list = clean)."""
    issues = []
    if "run_id" not in df.columns:
        df = df.assign(run_id=1)
    ok = df[df["error_message"] == ""]
    dup = ok.duplicated(["run_id", "question_id", "approach_name"], keep=False)
    if dup.any():
        issues.append(f"duplicate successful rows: {ok[dup][['run_id', 'question_id', 'approach_name']].values.tolist()}")
    qids = [q["question_id"] for q in qs]
    for run_id in sorted(df["run_id"].unique()):
        for a in APPROACHES:
            got = set(ok[(ok.run_id == run_id) & (ok.approach_name == a)].question_id)
            missing = sorted(set(qids) - got)
            if missing:
                issues.append(f"run {run_id} / {a}: no successful run for {missing}")
    if df["model_name"].nunique() != 1:
        issues.append(f"inconsistent model_name values: {df['model_name'].unique().tolist()}")
    for col in ["final_context_tokens", "input_prompt_tokens", "output_tokens", "total_latency_seconds"]:
        if ok[col].isna().any():
            issues.append(f"missing {col} in {int(ok[col].isna().sum())} successful rows")
    return issues


def per_question(df):
    """Per (question, pipeline): mean of each metric over the successful runs (the unit of analysis)."""
    ok = df[df["error_message"] == ""]
    cols = [m for m, _ in METRICS]
    pq = ok.groupby(["question_id", "approach_name"])[cols].mean()
    pq["n_runs"] = ok.groupby(["question_id", "approach_name"]).size()
    return pq.reset_index()


def summary(pq):
    """Mean/median over the 24 per-question values; paired Wilcoxon on per-question values."""
    try:
        from scipy.stats import wilcoxon
    except ImportError:
        wilcoxon = None
    b_all, o_all = pq[pq.approach_name == "baseline"].set_index("question_id"), pq[pq.approach_name == "optimized"].set_index("question_id")
    paired_ids = sorted(set(b_all.index) & set(o_all.index))
    rows = []
    for m, unit in METRICS:
        b, o = b_all[m].astype(float), o_all[m].astype(float)
        bm, bmed, om, omed = b.mean(), b.median(), o.mean(), o.median()
        pr_mean, pr_med = pct_reduction(bm, om), pct_reduction(bmed, omed)
        p, stat_note = None, ""
        if m in STRUCTURAL_ZERO_BASELINE:
            stat_note = "baseline has no such stage (structurally 0); test not meaningful"
        elif wilcoxon is not None and len(paired_ids) >= 6:
            d = (b.loc[paired_ids] - o.loc[paired_ids]).dropna()
            if len(d) >= 6 and (d != 0).any():
                p = float(wilcoxon(d).pvalue)
            else:
                stat_note = "all paired differences zero or n<6; test not applicable"
        if m in LOWER_IS_BETTER:
            interp = "baseline has no such stage; optimized adds this cost" if m in STRUCTURAL_ZERO_BASELINE else describe(pr_mean)
        else:
            interp = ("higher throughput" if pr_mean is not None and pr_mean < 0 else
                      "lower throughput" if pr_mean is not None and pr_mean > 0 else describe(pr_mean))
        rows.append({
            "metric": m, "unit": unit, "lower_is_better": m in LOWER_IS_BETTER,
            "aggregation": "per-question mean over runs, then mean/median across questions",
            "n_questions_baseline": int(b.notna().sum()), "n_questions_optimized": int(o.notna().sum()),
            "n_pairs": len(paired_ids), "runs_per_question_min": int(pq.n_runs.min()), "runs_per_question_max": int(pq.n_runs.max()),
            "baseline_mean": bm, "baseline_median": bmed, "baseline_std": b.std(),
            "optimized_mean": om, "optimized_median": omed, "optimized_std": o.std(),
            "pct_reduction_mean": pr_mean, "pct_reduction_median": pr_med,
            "mean_change_interpretation": interp,
            "wilcoxon_signed_rank_p_paired": p, "stat_note": stat_note,
        })
    return pd.DataFrame(rows)


def summary_by_run(df):
    """Each run analysed on its own - shows whether the conclusion is stable across repetitions."""
    ok = df[df["error_message"] == ""]
    rows = []
    for run_id in sorted(ok.run_id.unique()):
        d = ok[ok.run_id == run_id]
        for m in ["final_context_tokens", "input_prompt_tokens", "output_tokens", "total_tokens",
                  "ttft_seconds", "generation_time_seconds", "total_latency_seconds"]:
            b = d[d.approach_name == "baseline"][m].astype(float)
            o = d[d.approach_name == "optimized"][m].astype(float)
            rows.append({"run_id": run_id, "metric": m, "n_baseline": len(b), "n_optimized": len(o),
                         "baseline_mean": b.mean(), "optimized_mean": o.mean(),
                         "baseline_median": b.median(), "optimized_median": o.median(),
                         "pct_reduction_mean": pct_reduction(b.mean(), o.mean()),
                         "pct_reduction_median": pct_reduction(b.median(), o.median())})
    return pd.DataFrame(rows)


def reproducibility(df):
    """Across runs: are answers / contexts / token counts identical, and how much does latency vary?"""
    ok = df[df["error_message"] == ""]
    rows = []
    for (q, a), g in ok.groupby(["question_id", "approach_name"]):
        lat = g.total_latency_seconds.astype(float)
        rows.append({"question_id": q, "approach_name": a, "n_runs": len(g),
                     "answers_identical": g.generated_answer.nunique() == 1,
                     "final_context_identical": g.final_context.nunique() == 1,
                     "prompt_tokens_identical": g.input_prompt_tokens.nunique() == 1,
                     "output_tokens_identical": g.output_tokens.nunique() == 1,
                     "latency_mean_s": lat.mean(), "latency_sd_s": lat.std(ddof=1) if len(g) > 1 else None,
                     "latency_cv": (lat.std(ddof=1) / lat.mean()) if len(g) > 1 and lat.mean() > 0 else None,
                     "latency_min_s": lat.min(), "latency_max_s": lat.max()})
    rep = pd.DataFrame(rows)
    summ = {a: {"pairs": int((rep.approach_name == a).sum()),
                "answers_identical_across_runs": int(rep[rep.approach_name == a].answers_identical.sum()),
                "contexts_identical_across_runs": int(rep[rep.approach_name == a].final_context_identical.sum()),
                "output_tokens_identical_across_runs": int(rep[rep.approach_name == a].output_tokens_identical.sum()),
                "median_latency_cv": (float(rep[rep.approach_name == a].latency_cv.median())
                                      if rep[rep.approach_name == a].latency_cv.notna().any() else None)}
            for a in APPROACHES}
    return rep, summ


def automated_checks(df, qs):
    """Objective string/set checks only. These are NOT answer-quality scores."""
    ok = df[df["error_message"] == ""].copy()
    exp = {q["question_id"]: set(q["expected_documents"]) for q in qs}
    out = []
    for _, r in ok.iterrows():
        cited = json.loads(r["source_chunk_ids"]) if isinstance(r["source_chunk_ids"], str) else []
        final_docs = {c.split("-C")[0] for c in json.loads(r["final_chunks"])}
        out.append({
            "run_id": r.run_id, "question_id": r.question_id, "approach_name": r.approach_name, "answerable": r.answerable,
            "refusal_exact_match": str(r.refusal_exact_match).lower() == "true",
            "num_cited_ids": len(cited), "has_citation": len(cited) > 0,
            "cited_ids_subset_of_final_chunks": str(r.cited_ids_subset_of_final_chunks).lower() == "true",
            "expected_docs_all_in_final_context": exp[r.question_id].issubset(final_docs) if exp[r.question_id] else None,
            "finish_reason": r.get("finish_reason"),
        })
    chk = pd.DataFrame(out)
    agg = []
    for run_id in sorted(chk.run_id.unique()):
        for a in APPROACHES:
            c = chk[(chk.approach_name == a) & (chk.run_id == run_id)]
            ans, un = c[c.answerable], c[~c.answerable]
            agg.append({
                "run_id": run_id, "approach_name": a, "n_successful_runs": len(c),
                "unanswerable_refused_exact": int(un.refusal_exact_match.sum()), "unanswerable_total": len(un),
                "answerable_refused_exact": int(ans.refusal_exact_match.sum()), "answerable_total": len(ans),
                "answers_with_citation": int(c.has_citation.sum()),
                "answers_with_invalid_citation_ids": int((~c.cited_ids_subset_of_final_chunks).sum()),
                "answerable_with_all_expected_docs_in_context": int((ans.expected_docs_all_in_final_context == True).sum()),  # noqa: E712
                "responses_truncated_by_max_tokens": int((c.finish_reason == "length").sum()),
            })
    return chk, pd.DataFrame(agg)


def charts(df, pq, out):
    ok = df[df["error_message"] == ""]
    n_runs = int(ok.run_id.nunique())
    plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "axes.titleweight": "semibold",
                         "axes.labelcolor": INK2, "text.color": INK})
    agg_note = f"Unit: question (mean of {n_runs} run{'s' if n_runs > 1 else ''} per question). Bars: mean over questions; whiskers: ±1 SD."

    def bar_chart(metric, fname, title, ylabel):
        fig, ax = plt.subplots(figsize=(7, 4.6), facecolor=SURFACE)
        style_axes(ax)
        means, stds, ns = [], [], []
        for a in APPROACHES:
            v = pq[pq.approach_name == a][metric].astype(float).dropna()
            means.append(v.mean()); stds.append(v.std()); ns.append(len(v))
        x = np.arange(len(APPROACHES))
        bars = ax.bar(x, means, width=0.5, color=[COLORS[a] for a in APPROACHES], yerr=stds,
                      error_kw={"ecolor": INK2, "elinewidth": 1, "capsize": 4}, zorder=2)
        for bi, m in zip(bars, means):
            ax.annotate(f"{m:,.1f}" if m < 100 else f"{m:,.0f}", (bi.get_x() + bi.get_width() / 2, m),
                        xytext=(12, 4), textcoords="offset points", ha="left", color=INK, fontsize=10)
        ax.set_xticks(x, [f"{SHORT[a]}\n(n={n} questions)" for a, n in zip(APPROACHES, ns)])
        ax.set_ylabel(ylabel)
        ax.set_title(title, loc="left")
        ax.text(0, -0.2, agg_note, transform=ax.transAxes, color=INK2, fontsize=8)
        fig.tight_layout(); fig.savefig(out / fname, dpi=160); plt.close(fig)

    bar_chart("final_context_tokens", "average_final_context_tokens.png",
              "Average context tokens sent to the LLM per question", "Final context tokens (Gemma tokenizer)")
    bar_chart("total_tokens", "average_total_tokens.png",
              "Average total tokens per question (prompt + output)", "Total tokens (LLM server usage)")

    # stacked stage means
    stages = [("retrieval_time_seconds", "Retrieval"), ("reranking_time_seconds", "Reranking"),
              ("compression_time_seconds", "Compression"), ("generation_time_seconds", "LLM generation")]
    fig, ax = plt.subplots(figsize=(7.5, 4.8), facecolor=SURFACE)
    style_axes(ax)
    x = np.arange(len(APPROACHES)); bottom = np.zeros(len(APPROACHES))
    for (col, lab), colr in zip(stages, STAGE_COLORS):
        vals = np.array([pq[pq.approach_name == a][col].astype(float).mean() for a in APPROACHES])
        ax.bar(x, vals, bottom=bottom, width=0.5, color=colr, label=lab, edgecolor=SURFACE, linewidth=1, zorder=2)
        bottom += vals
    totals = [pq[pq.approach_name == a]["total_latency_seconds"].astype(float).mean() for a in APPROACHES]
    pre = pq[pq.approach_name == "optimized"][["retrieval_time_seconds", "reranking_time_seconds", "compression_time_seconds"]].sum(axis=1).mean()
    for xi, t in zip(x, totals):
        ax.annotate(f"total {t:.2f} s", (xi, t), xytext=(0, 5), textcoords="offset points", ha="center", color=INK)
    ax.annotate(f"retrieval+rerank+compress = {pre:.2f} s", (1, 0), xytext=(0, -30), textcoords="offset points",
                ha="center", color=INK2, fontsize=8, annotation_clip=False)
    ax.set_xticks(x, [SHORT[a] for a in APPROACHES])
    ax.set_ylabel("Mean latency (seconds)")
    ax.set_title("Average end-to-end latency per question, by pipeline stage", loc="left")
    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    ax.set_ylim(0, max(totals) * 1.15)
    ax.text(0, -0.2, "Stacked stage means (per-question means over runs); label = mean measured total latency.",
            transform=ax.transAxes, color=INK2, fontsize=8)
    fig.tight_layout(); fig.savefig(out / "average_latency.png", dpi=160); plt.close(fig)

    # scatter: every successful run
    fig, ax = plt.subplots(figsize=(7, 4.8), facecolor=SURFACE)
    style_axes(ax, xgrid=True)
    for a, mk in zip(APPROACHES, ["o", "s"]):
        d = ok[ok.approach_name == a]
        ax.scatter(d["final_context_tokens"], d["total_latency_seconds"], s=36, marker=mk, color=COLORS[a],
                   edgecolor=SURFACE, linewidth=1.2, alpha=0.9, label=LABELS[a], zorder=3)
    ax.set_xlabel("Final context tokens (Gemma tokenizer)")
    ax.set_ylabel("Total latency (seconds)")
    ax.set_title(f"Context size vs. end-to-end latency ({len(ok)} runs: questions × pipelines × {n_runs} runs)", loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8.5, loc="upper left")
    fig.tight_layout(); fig.savefig(out / "token_latency_scatter.png", dpi=160); plt.close(fig)

    # per-question context tokens
    qids = sorted(pq.question_id.unique())
    fig, ax = plt.subplots(figsize=(12, 4.8), facecolor=SURFACE)
    style_axes(ax)
    x = np.arange(len(qids)); w = 0.4
    for i, a in enumerate(APPROACHES):
        d = pq[pq.approach_name == a].set_index("question_id")["final_context_tokens"].reindex(qids).astype(float)
        ax.bar(x + (i - 0.5) * w, d.values, width=w - 0.03, color=COLORS[a], label=LABELS[a], zorder=2)
    ax.set_xticks(x, qids, fontsize=8)
    ax.set_xlabel("Question ID (Q21–Q24 are intentionally unanswerable)")
    ax.set_ylabel("Final context tokens")
    ax.set_title("Context tokens sent to the LLM, per question", loc="left")
    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0, -0.14), ncol=2)
    fig.tight_layout(); fig.savefig(out / "per_question_token_comparison.png", dpi=160); plt.close(fig)

    # per-question latency, every run shown (run-to-run variability)
    fig, ax = plt.subplots(figsize=(12, 4.8), facecolor=SURFACE)
    style_axes(ax)
    for i, a in enumerate(APPROACHES):
        d = ok[ok.approach_name == a]
        xs = np.array([qids.index(q) for q in d.question_id]) + (i - 0.5) * 0.3
        ax.scatter(xs, d.total_latency_seconds, s=26, color=COLORS[a], edgecolor=SURFACE, linewidth=1, zorder=3,
                   label=f"{LABELS[a]} (each run)")
        m = pq[pq.approach_name == a].set_index("question_id")["total_latency_seconds"].reindex(qids)
        ax.scatter(np.arange(len(qids)) + (i - 0.5) * 0.3, m.values, marker="_", s=260, color=INK, linewidth=1.5, zorder=4,
                   label="Per-question mean" if i == 0 else None)
    ax.set_xticks(np.arange(len(qids)), qids, fontsize=8)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Question ID")
    ax.set_ylabel("Total latency (seconds)")
    ax.set_title(f"Total latency per question across {n_runs} runs (run-to-run variability)", loc="left")
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(0, -0.14), ncol=3)
    fig.tight_layout(); fig.savefig(out / "latency_per_question_runs.png", dpi=160); plt.close(fig)


def manual_template(df, qs, rep, out):
    """Answers from the first successful run; scoring columns left BLANK. Never overwrites filled-in scores."""
    ok = df[df["error_message"] == ""].sort_values("run_id")
    target = out / "manual_evaluation_template.csv"
    if target.exists():
        old = pd.read_csv(target, keep_default_na=False)
        if any((old.get(c, pd.Series(dtype=str)).astype(str).str.strip() != "").any() for c in SCORE_COLUMNS + ["notes"]):
            target = out / "manual_evaluation_template_NEW.csv"
            print(f"NOTE: existing manual scores found - not overwritten; new template written to {target.name}")
    same = rep.set_index(["question_id", "approach_name"])["answers_identical"]
    rows = []
    for q in qs:
        def pick(a, col):
            r = ok[(ok.question_id == q["question_id"]) & (ok.approach_name == a)]
            return r.iloc[0][col] if len(r) else "RUN FAILED - see experiment_results.csv"
        rows.append({
            "question_id": q["question_id"], "question": q["question"], "category": q["category"],
            "answerable": q["answerable"], "reference_answer": q["reference_answer"],
            "baseline_answer": pick("baseline", "generated_answer"), "optimized_answer": pick("optimized", "generated_answer"),
            "baseline_answer_identical_across_runs": bool(same.get((q["question_id"], "baseline"), False)),
            "optimized_answer_identical_across_runs": bool(same.get((q["question_id"], "optimized"), False)),
            "baseline_final_context": pick("baseline", "final_context"), "optimized_final_context": pick("optimized", "final_context"),
            **{c: "" for c in SCORE_COLUMNS}, "notes": "",
        })
    pd.DataFrame(rows).to_csv(target, index=False, encoding="utf-8-sig")
    return target


def analyze_results(results_dir=None):
    out = Path(results_dir) if results_dir else config.RESULTS_DIR
    qs = json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))
    df = load(out)
    issues = validate(df, qs)
    pq = per_question(df)
    pq.to_csv(out / "per_question_means.csv", index=False)
    s = summary(pq); s.to_csv(out / "summary_metrics.csv", index=False)
    sbr = summary_by_run(df); sbr.to_csv(out / "summary_metrics_by_run.csv", index=False)
    rep, rep_summ = reproducibility(df); rep.to_csv(out / "reproducibility_per_question.csv", index=False)
    chk, agg = automated_checks(df, qs)
    chk.to_csv(out / "automated_checks_per_run.csv", index=False)
    agg.to_csv(out / "automated_checks_summary.csv", index=False)
    ok = df[df["error_message"] == ""]
    tokcheck = ok["input_prompt_tokens"].astype(float) - ok["local_prompt_tokens_estimate"].astype(float)
    import scipy
    info = {"rows": len(df), "runs": sorted(int(r) for r in df.run_id.unique()),
            "successful": int((df.error_message == "").sum()), "failed": int((df.error_message != "").sum()),
            "retries": int(pd.to_numeric(df.get("retry_count"), errors="coerce").fillna(0).sum()),
            "validation_issues": issues,
            "server_minus_local_prompt_tokens": {"mean": float(tokcheck.mean()), "min": float(tokcheck.min()), "max": float(tokcheck.max())},
            "token_count_sources": ok["token_count_source"].value_counts().to_dict(),
            "compression_budget_hit_runs": int((ok[ok.approach_name == "optimized"].compression_budget_hit.astype(str) == "True").sum()),
            "reproducibility": rep_summ,
            "analysis_environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                                     "pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
                                     "matplotlib": matplotlib.__version__,
                                     "wilcoxon_settings": "scipy.stats.wilcoxon defaults on per-question means (two-sided, zero_method='wilcox', method='auto')"}}
    (out / "analysis_checks.json").write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")
    charts(df, pq, out)
    manual_template(df, qs, rep, out)
    print(json.dumps({k: info[k] for k in ["rows", "runs", "successful", "failed", "retries", "validation_issues",
                                           "server_minus_local_prompt_tokens", "reproducibility"]}, indent=2, default=str))
    return {"df": df, "pq": pq, "summary": s, "by_run": sbr, "reproducibility": rep, "checks": agg, "info": info}

## 13. Evidence coverage (retrieval/compression diagnostic)
This checks whether the verbatim source phrases that support each reference answer reached the model. It compares the baseline context, the optimized top-3 before compression, and the optimized context after compression.
This separates evidence lost at **reranking** from evidence lost at **compression**. It is a diagnostic, not a quality score.

In [ ]:
"""Evidence coverage: a retrieval/compression DIAGNOSTIC, not an answer-quality score.

For each answerable question, EVIDENCE lists short verbatim substrings of the source documents that together
support the reference answer in data/eval_questions.json. The reference answers were fixed before any run;
these substrings map each reference-answer fact to its verbatim source sentence. (They were written after the
first, pilot run - see the report - without regard to which pipeline contained them.)
"""
EVIDENCE = {
    "Q01": ["1.75 days of paid annual leave", "21 days per leave year"],
    "Q02": ["return window of 14 days", "15% restocking fee", "No restocking fee is charged for defective electronics"],
    "Q03": ["at least 14 characters long"],
    "Q04": ["core hours, which are 10:00 AM to 3:00 PM, Monday to Friday"],
    "Q05": ["Priority 1 (Critical)", "Response within 15 minutes; resolution target of 4 hours"],
    "Q06": ["one-time home office allowance of $300", "claimed within 60 days"],
    "Q07": ["clearance items marked \"Final Sale\", gift cards, personalized or engraved items, perishable food, and opened hygiene products"],
    "Q08": ["primary caregiver receives 16 weeks of leave at full pay", "at least 6 months of continuous service"],
    "Q09": ["1. Restart the terminal", "2. Check that the network cable", "3. Move customers to a working backup terminal",
            "Log it as Priority 1 if more than 2 terminals"],
    "Q10": ["1. Acknowledge the customer's concern", "2. Collect the order number", "3. Log the case in CaseTrack within 30 minutes",
            "4. Assign the case to the correct tier", "5. Give the customer the case number", "6. When the case is resolved"],
    "Q11": ["submitted at least 10 working days before the first day of leave", "within 3 working days of submission"],
    "Q12": ["locked for 30 minutes", "call the Service Desk on extension 4357, provide their employee ID",
            "identity verified by their line manager"],
    "Q13": ["after successfully completing their 3-month probation period", "\"Flexible Work Request\" form in the PeopleHub portal",
            "respond to the request within 10 working days"],
    "Q14": ["submit a Software Request ticket in HelpPoint", "security review, which can take up to 10 business days",
            "only if the security review is passed"],
    "Q15": ["report it to the Service Desk within 1 hour", "report it to Information Security as a security incident",
            "reported to Information Security within 1 hour of discovery"],
    "Q16": ["escalated to Tier 3 if the exception value is above $200 and up to $1,000",
            "Tier 3 must acknowledge an escalated case within 1 business day"],
    "Q17": ["connect through the NorthLink VPN", "must not work on Confidential or Restricted data in public places"],
    "Q18": ["must be reported immediately to Information Security", "within 1 hour of discovery"],
    "Q19": ["mandatory for company email, the HelpPoint portal, the PeopleHub portal, and the NorthLink VPN",
            "reset a forgotten password themselves through the self-service option in the HelpPoint portal"],
    "Q20": ["return window of 14 days", "15% restocking fee", "goodwill credit of up to $50"],
}


def _norm_ev(s):
    return " ".join(str(s).replace("’", "'").split()).lower()


def evidence_coverage(results_dir=None):
    """Evidence phrases present in context: baseline, optimized before compression, optimized after compression.
    Contexts are deterministic across runs (checked in reproducibility), so the first run is used."""
    results_dir = Path(results_dir) if results_dir else config.RESULTS_DIR
    ROOT = config.ROOT
    df = pd.read_csv(results_dir / "experiment_results.csv", keep_default_na=False)
    if "run_id" not in df.columns:
        df["run_id"] = 1
    df = df[df.error_message == ""]
    df = df[df.run_id.astype(int) == df.run_id.astype(int).min()]
    chunks = {c["chunk_id"]: c["text"] for c in json.loads((ROOT / "index" / "chunks.json").read_text(encoding="utf-8"))}
    docs = _norm_ev(" ".join(p.read_text(encoding="utf-8") for p in (ROOT / "data" / "documents").glob("*.md")))
    for q, evs in EVIDENCE.items():
        for e in evs:
            assert _norm_ev(e) in docs, f"evidence string not verbatim in documents: {q}: {e}"
    rows = []
    for _, r in df.iterrows():
        q = r.question_id
        if q not in EVIDENCE:
            continue
        final = _norm_ev(r.final_context)
        pre = _norm_ev("\n".join(chunks[c] for c in json.loads(r.final_chunks)))
        for e in EVIDENCE[q]:
            rows.append({"question_id": q, "approach_name": r.approach_name, "evidence": e,
                         "in_pre_compression_context": _norm_ev(e) in pre, "in_final_context": _norm_ev(e) in final})
    ev = pd.DataFrame(rows)
    ev.to_csv(results_dir / "evidence_coverage.csv", index=False)
    per_q = ev.groupby(["question_id", "approach_name"]).agg(
        n_evidence=("evidence", "size"), in_pre=("in_pre_compression_context", "sum"), in_final=("in_final_context", "sum")).reset_index()
    per_q["all_evidence_in_final"] = per_q.in_final == per_q.n_evidence
    summ = []
    for a in ["baseline", "optimized"]:
        e, p = ev[ev.approach_name == a], per_q[per_q.approach_name == a]
        summ.append({"approach_name": a, "evidence_items": len(e),
                     "evidence_items_in_pre_compression_context": int(e.in_pre_compression_context.sum()),
                     "evidence_items_in_final_context": int(e.in_final_context.sum()),
                     "questions_with_all_evidence_in_final_context": int(p.all_evidence_in_final.sum()),
                     "answerable_questions": len(p)})
    pd.DataFrame(summ).to_csv(results_dir / "evidence_coverage_summary.csv", index=False)
    per_q.to_csv(results_dir / "evidence_coverage_per_question.csv", index=False)
    evidence_chart(per_q, results_dir)
    print(pd.DataFrame(summ).to_string())
    return per_q, pd.DataFrame(summ)


def evidence_chart(per_q, results_dir):
    """Per-question share of evidence phrases present: baseline context, optimized top-3 before
    compression, optimized context after compression. Drawn only from per_q (computed above)."""
    INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
    series = [("baseline", "in_final", "Baseline: context sent to LLM (top-5 full chunks)", "#2a78d6"),
              ("optimized", "in_pre", "Token-efficient: top-3 after reranking, before compression", "#eb6834"),
              ("optimized", "in_final", "Token-efficient: context sent to LLM (after compression)", "#1baf7a")]
    qids = sorted(per_q.question_id.unique())
    fig, ax = plt.subplots(figsize=(12, 5.2), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["left", "bottom"]:
        ax.spines[sp].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=8.5)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    x = np.arange(len(qids))
    w = 0.27
    totals = {}
    for i, (appr, col, label, color) in enumerate(series):
        d = per_q[per_q.approach_name == appr].set_index("question_id").reindex(qids)
        pct = d[col] / d["n_evidence"] * 100
        totals[label] = (int(d[col].sum()), int(d["n_evidence"].sum()))
        ax.bar(x + (i - 1) * w, pct.values, width=w - 0.03, color=color, zorder=2,
               label=f"{label}  [{totals[label][0]}/{totals[label][1]} phrases]")
    ax.set_xticks(x, qids)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Evidence phrases present (%)", color=INK2)
    ax.set_xlabel("Answerable question ID", color=INK2)
    ax.set_title("Did the supporting evidence reach the LLM? (retrieval diagnostic, not a quality score)",
                 loc="left", fontsize=12, fontweight="semibold", color=INK)
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=1)
    fig.tight_layout()
    fig.savefig(results_dir / "evidence_coverage.png", dpi=160)
    plt.close(fig)

## 14. Latency decomposition (exploratory)
Two pooled linear fits: TTFT against prompt tokens (the cost of processing the prompt) and decode time against output tokens (the cost of writing the answer).
The fits show how much of the latency difference comes from the smaller context and how much from shorter answers.

In [ ]:
def ols(x, y):
    """Ordinary least squares y = a + b*x; returns intercept, slope and R^2."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    A = np.vstack([np.ones_like(x), x]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    pred = A @ coef
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return {"intercept_s": float(coef[0]), "slope_s_per_token": float(coef[1]), "r2": float(r2), "n": int(len(x))}


def latency_decomposition(results_dir=None):
    """Exploratory: pooled OLS fits over every successful run, and paired per-question comparisons
    (per-question means over runs). Writes results/latency_decomposition.json and latency_components.png."""
    R = Path(results_dir) if results_dir else config.RESULTS_DIR
    df = load(R)
    df = df[df.error_message == ""].copy()
    df["decode"] = df.generation_time_seconds - df.ttft_seconds
    pq = per_question(df)
    b = pq[pq.approach_name == "baseline"].set_index("question_id")
    o = pq[pq.approach_name == "optimized"].set_index("question_id")
    ids = sorted(set(b.index) & set(o.index))
    d = b.loc[ids, "total_latency_seconds"] - o.loc[ids, "total_latency_seconds"]
    pre_llm = o.retrieval_time_seconds + o.reranking_time_seconds + o.compression_time_seconds
    # per-run wins: in how many (run, question) pairs was the optimized pipeline faster?
    wide = df.pivot_table(index=["run_id", "question_id"], columns="approach_name", values="total_latency_seconds")
    wide = wide.dropna()
    out = {
        "ttft_vs_input_prompt_tokens": ols(df.input_prompt_tokens, df.ttft_seconds),
        "decode_time_vs_output_tokens": ols(df.output_tokens, df.decode),
        "paired_questions": len(ids),
        "questions_optimized_lower_mean_latency": int((d > 0).sum()),
        "questions_optimized_higher_mean_latency": int((d < 0).sum()),
        "paired_mean_latency_difference_s(baseline-optimized)": {"mean": float(d.mean()), "median": float(d.median()),
                                                                  "min": float(d.min()), "max": float(d.max())},
        "run_level_pairs": int(len(wide)),
        "run_level_pairs_optimized_faster": int((wide["baseline"] > wide["optimized"]).sum()),
        "questions_optimized_fewer_output_tokens": int((b.loc[ids, "output_tokens"] > o.loc[ids, "output_tokens"]).sum()),
        "questions_equal_output_tokens": int((b.loc[ids, "output_tokens"] == o.loc[ids, "output_tokens"]).sum()),
        "optimized_pre_llm_overhead_s(retrieval+rerank+compress)": {"mean": float(pre_llm.mean()), "max": float(pre_llm.max())},
        "ttft_paired_difference_s(baseline-optimized)": {
            "mean": float((b.loc[ids, "ttft_seconds"] - o.loc[ids, "ttft_seconds"]).mean()),
            "questions_optimized_lower": int((b.loc[ids, "ttft_seconds"] > o.loc[ids, "ttft_seconds"]).sum())},
        "note": "Exploratory; OLS pooled over both pipelines and all runs; paired comparisons use per-question means over runs.",
    }
    (R / "latency_decomposition.json").write_text(json.dumps(out, indent=2), encoding="utf-8")

    # chart: two small multiples on separate axes (no dual y-axis)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), facecolor=SURFACE)
    panels = [("input_prompt_tokens", "ttft_seconds", "ttft_vs_input_prompt_tokens", "Prompt tokens (LLM server usage)",
               "Time to first token (s)", "A. Prompt processing: TTFT vs prompt length"),
              ("output_tokens", "decode", "decode_time_vs_output_tokens", "Output tokens (LLM server usage)",
               "Decode time = generation − TTFT (s)", "B. Answer writing: decode time vs answer length")]
    for ax, (xc, yc, fitkey, xl, yl, title) in zip(axes, panels):
        style_axes(ax, xgrid=True)
        for a, mk in [("baseline", "o"), ("optimized", "s")]:
            dd = df[df.approach_name == a]
            ax.scatter(dd[xc], dd[yc], s=34, marker=mk, color=COLORS[a], edgecolor=SURFACE, linewidth=1.2, zorder=3,
                       label=SHORT[a] + " RAG")
        fit = out[fitkey]
        xs = np.linspace(0, df[xc].max() * 1.05, 50)
        ax.plot(xs, fit["intercept_s"] + fit["slope_s_per_token"] * xs, color=INK2, linewidth=1.2, linestyle="--", zorder=2,
                label=f"OLS (all runs): {fit['intercept_s']:.2f} s + {fit['slope_s_per_token'] * 1000:.1f} ms/token, R² = {fit['r2']:.3f}")
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(title, loc="left", fontsize=11)
        ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        ax.legend(frameon=False, fontsize=8, loc="upper left")
    fig.suptitle(f"Where the latency goes ({len(df)} runs)", x=0.01, ha="left", fontsize=12.5, fontweight="semibold", color=INK)
    fig.tight_layout(); fig.savefig(R / "latency_components.png", dpi=160); plt.close(fig)
    print(json.dumps(out, indent=2))
    return out

## 15. Replication against the pilot run
An earlier complete run (28 Sep 2026, one pass, same pipeline code executed as scripts) is preserved unchanged in `results/pilot_run_2026-09-28/`.
It is compared with the current results to check whether the findings replicate.

In [ ]:
PILOT_DIR_NAME = "pilot_run_2026-09-28"   # earlier complete run (same code as .py scripts, 1 run), kept unchanged


def replication_vs_pilot(results_dir=None):
    """Compares the current results with the preserved pilot run, if present. Writes replication_vs_pilot.csv."""
    R = Path(results_dir) if results_dir else config.RESULTS_DIR
    pilot_csv = R / PILOT_DIR_NAME / "experiment_results.csv"
    if not pilot_csv.exists():
        print("No pilot run found - replication comparison skipped.")
        return None
    cur, pil = load(R), load(R / PILOT_DIR_NAME)
    cur_pq, pil_pq = per_question(cur), per_question(pil)
    rows = []
    for m in ["final_context_tokens", "input_prompt_tokens", "output_tokens", "total_tokens",
              "ttft_seconds", "generation_time_seconds", "total_latency_seconds"]:
        r = {"metric": m}
        for tag, pq in [("pilot", pil_pq), ("current", cur_pq)]:
            for a in APPROACHES:
                r[f"{tag}_{a}_mean"] = pq[pq.approach_name == a][m].mean()
            r[f"{tag}_pct_reduction_mean"] = pct_reduction(r[f"{tag}_baseline_mean"], r[f"{tag}_optimized_mean"])
        rows.append(r)
    rep = pd.DataFrame(rows)
    rep.to_csv(R / "replication_vs_pilot.csv", index=False)

    # identity of answers / contexts: pilot vs every current run
    ok_c, ok_p = cur[cur.error_message == ""], pil[pil.error_message == ""]
    key = ["question_id", "approach_name"]
    m = ok_c.merge(ok_p[key + ["generated_answer", "final_context", "output_tokens"]], on=key, suffixes=("", "_pilot"))
    ident = {
        "pairs_compared": int(len(m)),
        "answers_identical_to_pilot": int((m.generated_answer == m.generated_answer_pilot).sum()),
        "contexts_identical_to_pilot": int((m.final_context == m.final_context_pilot).sum()),
        "output_tokens_identical_to_pilot": int((m.output_tokens == m.output_tokens_pilot).sum()),
    }
    diff = m[m.generated_answer != m.generated_answer_pilot][["run_id"] + key].values.tolist()
    ident["answers_differing_from_pilot"] = diff
    (R / "replication_vs_pilot.json").write_text(json.dumps(ident, indent=2, default=str), encoding="utf-8")
    print(rep.round(3).to_string(index=False))
    print(json.dumps({k: v for k, v in ident.items() if k != "answers_differing_from_pilot"}, indent=2))
    return rep, ident

## 16. Report builder
Builds `results/experiment_report.md` entirely from the result files. Failure cases are detected by objective criteria. The author's qualitative notes are added only if `results/author_notes.md` exists; they are written after reading the recorded answers.

In [ ]:
def _fmt(x, d=2):
    try:
        return "n/a" if x is None or pd.isna(x) else f"{x:,.{d}f}"
    except TypeError:
        return str(x)


def failure_table(results_dir):
    """Automatically detected cases (objective criteria only) for the first successful run of each pair."""
    df = load(results_dir)
    ok = df[df.error_message == ""].sort_values("run_id")
    first = ok.groupby(["question_id", "approach_name"]).head(1)
    evq = pd.read_csv(results_dir / "evidence_coverage_per_question.csv")
    ev = evq.set_index(["question_id", "approach_name"])
    rows = []
    for _, r in first.iterrows():
        flags = []
        refused = str(r.refusal_exact_match) == "True"
        if r.answerable and refused:
            flags.append("false refusal (answerable question)")
        if (not r.answerable) and not refused:
            flags.append("unanswerable question NOT refused")
        if str(r.cited_ids_subset_of_final_chunks) != "True":
            flags.append(f"cites chunk ID(s) not in its context: {r.source_chunk_ids}")
        if r.answerable and not refused and json.loads(r.source_chunk_ids) == []:
            flags.append("no chunk-ID citation")
        if (r.question_id, r.approach_name) in ev.index:
            e = ev.loc[(r.question_id, r.approach_name)]
            if e.in_final < e.n_evidence:
                stage = ""
                if r.approach_name == "optimized":
                    lost_rerank, lost_comp = int(e.n_evidence - e.in_pre), int(e.in_pre - e.in_final)
                    stage = f" (missing after rerank: {lost_rerank}, removed by compression: {lost_comp})"
                flags.append(f"evidence in context {int(e.in_final)}/{int(e.n_evidence)}{stage}")
        if flags:
            rows.append({"question_id": r.question_id, "approach": r.approach_name, "issues": "; ".join(flags),
                         "answer": r.generated_answer.replace("\n", " ").replace("|", "/").strip()})
    return pd.DataFrame(rows)


def build_report(results_dir=None):
    R = Path(results_dir) if results_dir else config.RESULTS_DIR
    rj = lambda n: json.loads((R / n).read_text(encoding="utf-8"))
    meta, ana, lat = rj("experiment_metadata.json"), rj("analysis_checks.json"), rj("latency_decomposition.json")
    s = pd.read_csv(R / "summary_metrics.csv").set_index("metric")
    sbr = pd.read_csv(R / "summary_metrics_by_run.csv")
    chk = pd.read_csv(R / "automated_checks_summary.csv")
    ev = pd.read_csv(R / "evidence_coverage_summary.csv").set_index("approach_name")
    df = load(R)
    qs = json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))
    man = json.loads((config.INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
    warm = rj("warmup_log.json")
    pre = rj("preflight.json") if (R / "preflight.json").exists() else None
    itest = rj("integration_tests.json") if (R / "integration_tests.json").exists() else None
    otest = rj("offline_tests.json") if (R / "offline_tests.json").exists() else None
    notes_path = R / "author_notes.md"
    hw = meta["hardware_os"]
    rt = meta.get("llm_runtime") or {"name": "LM Studio (desktop app)"}
    is_lms = rt["name"].startswith("LM Studio")
    rt_name = "LM Studio" if is_lms else rt["name"]
    gpu_str = hw.get("nvidia_smi") or ", ".join(hw.get("gpu_names", [])) or "none detected"
    where = (f"{rt_name} on {hw.get('cpu_name', 'the test laptop')}" if is_lms
             else f"{rt_name}, GPU: {gpu_str}")
    n_runs = len(ana["runs"])
    rep_s = ana["reproducibility"]
    fs = lambda m, col, d=1: _fmt(s.loc[m, col], d)

    def row(m, d):
        r = s.loc[m]
        p = r.wilcoxon_signed_rank_p_paired
        return (f"| `{m}` | {_fmt(r.baseline_mean, d)} | {_fmt(r.baseline_median, d)} | {_fmt(r.optimized_mean, d)} | "
                f"{_fmt(r.optimized_median, d)} | {_fmt(r.pct_reduction_mean, 1)} | {_fmt(r.pct_reduction_median, 1)} | "
                f"{r.mean_change_interpretation} | {'n/a' if pd.isna(p) else f'{p:.2g}'} |")

    def per_run(a, col):
        c = chk[chk.approach_name == a].sort_values("run_id")
        return ", ".join(str(int(v)) for v in c[col])

    T = []
    w = T.append
    w("# Token-Efficient RAG: Reducing Context Size and Latency Through Reranking and Contextual Compression\n")
    w("## Experiment report\n")
    w(f"*Experiment started (UTC): {meta['timestamp_utc']}. Executed from `{meta.get('executed_from', 'n/a')}`. "
      f"Every number in this report was read from the files in `results/` when the report was built; nothing was estimated or typed in by hand. "
      "Manual answer-quality scoring is **not** included (see Section 13).*\n")

    w("## 1. Research question\n")
    w(f"> {meta['research_question']}\n")

    w("## 2. Experimental setup\n")
    w("Two RAG pipelines were run on the same questions, documents, chunks, embeddings, FAISS index, LLM, system prompt, user template and generation parameters. "
      "Only the retrieval, reranking and compression strategy differed.\n")
    w("| Setting | Baseline RAG | Token-efficient RAG |\n|---|---|---|")
    w(f"| Initial retrieval | FAISS top-{meta['baseline']['top_k']} | FAISS top-{meta['optimized']['initial_top_k']} |")
    w(f"| Reranking | none | `{meta['reranker_model']}` → top-{meta['optimized']['final_top_k_after_rerank']} |")
    w(f"| Context | full chunk text | extractive sentence compression, ≤ {meta['optimized']['compression_token_target']} tokens |")
    w(f"| LLM | `{meta['llm']['model_id']}` | same |")
    w(f"| Generation parameters | `{json.dumps(meta['llm']['generation_params'])}`, streaming | same |")
    w("| System prompt | identical (below) | identical |\n")
    w(f"System prompt used by both pipelines:\n\n> {meta['prompt']['system_prompt']}\n")
    w(f"User-message template: `{meta['prompt']['user_template']!r}`. In the context, each chunk (or group of compressed sentences) is headed by its chunk ID, e.g. `[LEAVE-C01]`.\n")
    w(f"**Repetitions and order.** The experiment was run **{n_runs} time(s)**, each a complete pass over all 24 questions with both pipelines "
      f"({len(df)} pipeline runs in total). {meta['run_order']}. "
      f"**Warm-up:** before measurement began, one question from outside the evaluation set (\"{warm['question']}\") was run through both pipelines. "
      "It is logged separately in `warmup_log.json` and is not included in any result.\n")
    w("**Unit of analysis.** The unit of analysis is the question. For each question and pipeline, every metric is averaged over the runs; means, medians and paired tests are then computed over the 24 questions. "
      "Each run on its own is reported in Section 10.2.\n")

    w("## 3. Hardware and software\n")
    ni = meta["llm"]["lm_studio_native_model_info"]
    w("| Item | Value |\n|---|---|")
    w(f"| OS | {hw.get('platform')} |")
    w(f"| CPU | {hw.get('cpu_name', hw.get('processor'))} ({hw.get('cpu_physical_cores')} cores / {hw.get('cpu_logical_cores')} threads) |")
    if is_lms:
        w(f"| GPU | {gpu_str} (whether LM Studio offloaded model layers to the GPU was not recorded, so this is **unknown**) |")
    else:
        w(f"| GPU | {gpu_str} (all model layers offloaded: `-ngl 999`) |")
    w(f"| RAM | {hw.get('ram_total_gb')} GB |")
    w(f"| Python | {meta['python_version'].split()[0]} |")
    if is_lms:
        w(f"| LLM runtime | LM Studio, OpenAI-compatible API at `{meta['llm']['base_url']}` (OpenAI SDK used only as a local client) |")
    else:
        w(f"| LLM runtime | {rt_name}, llama.cpp commit `{rt.get('llama_cpp_commit', 'n/a')[:12]}`, {rt.get('build')}; OpenAI-compatible API at `{meta['llm']['base_url']}` (OpenAI SDK used only as a local client); server args `{' '.join(rt.get('server_args', []))}` |")
    if is_lms:
        w(f"| Model | `{ni.get('id')}`, arch={ni.get('arch')}, quantization={ni.get('quantization')}, publisher={ni.get('publisher')}, "
          f"loaded context={ni.get('loaded_context_length')} (as reported by LM Studio); repository {meta['llm']['model_repository_reported_by_user']} |")
    else:
        w(f"| Model | served as `{meta['llm']['model_id']}`; file `{rt.get('model_file')}` from {rt.get('model_repository')} "
          f"(SHA-256 `{str(rt.get('model_file_sha256'))[:16]}…`, {rt.get('model_file_bytes', 0) / 1e9:.2f} GB), Q4_K_M |")
    w(f"| Embeddings | `{meta['embedding_model']}` (dim {man['embedding_dim']}, {meta['local_encoder_device']}) |")
    w(f"| Reranker | `{meta['reranker_model']}` ({meta['local_encoder_device']}) |")
    w(f"| Vector store | FAISS {man['faiss_version']}, {man['faiss_index_type']} |")
    w(f"| Tokenizer | {meta['token_counting']['local_tokenizer']} |")
    w("| Packages | " + ", ".join(f"{k} {v}" for k, v in meta["software_versions"].items() if v) + " |\n")

    w("## 4. Dataset\n")
    cats = pd.Series([q["category"] for q in qs]).value_counts().to_dict()
    w("The corpus is six original, internally consistent documents for a fictional company, *Northstar Retail Systems*: leave policy, IT support procedures, product returns, "
      "customer-support escalation, information-security rules, and remote work / office hours (`data/documents/`). "
      f"The chunking settings were {man['chunk_size']} characters with {man['chunk_overlap']} characters of overlap, with cuts at word boundaries; this produced **{man['num_chunks']} chunks**. "
      "Each chunk keeps its document ID, chunk ID and character offsets (`index/chunks.json`).\n")
    w("There are 24 evaluation questions (`data/eval_questions.json`), each with a reference answer written before any run. By category: " +
      ", ".join(f"{k}: {v}" for k, v in cats.items()) +
      ". For the four unanswerable questions (Q21–Q24), the expected response is exactly *\"I cannot find the answer in the provided documents.\"*\n")

    w("## 5. Baseline methodology\n")
    w("1. Embed the question (all-MiniLM-L6-v2, L2-normalised).\n2. Retrieve the top 5 chunks by exact FAISS inner-product (cosine) search.\n"
      "3. Concatenate the **full text** of the 5 chunks, each under its `[CHUNK-ID]`.\n4. Send the system prompt and user message to Gemma 3 4B through the local OpenAI-compatible server.\n")
    w("## 6. Optimized (token-efficient) methodology\n")
    w("1. Same query embedding and FAISS index; retrieve the top 10 chunks.\n2. Rerank the 10 with the cross-encoder and keep the top 3.\n"
      "3. Extractive compression of the top 3 (Section 8).\n4. Same system prompt, user template, model and parameters, with the compressed context.\n")
    w("## 7. Reranking method\n")
    w("`cross-encoder/ms-marco-MiniLM-L-6-v2` scores each (question, chunk) pair from the 10 FAISS candidates. Chunks are sorted by score, with ties broken by FAISS rank, and the top 3 are kept. "
      "Every run's scores are stored in `reranker_scores`.\n")
    w("## 8. Compression method\n")
    w(f"{meta['optimized']['compression_method']}.\n\n"
      "1. Each of the top-3 chunks is split into lines, then into sentences, using a regular expression; list and heading markers (e.g. `1.`) stay attached to their text.\n"
      "2. Duplicates caused by the chunk overlap are removed.\n"
      "3. Every remaining segment is scored against the question by the same cross-encoder.\n"
      f"4. Segments are kept in descending score order while their logit is ≥ {meta['optimized']['compression_score_threshold_logit']}, and always until at least {meta['optimized']['compression_min_segments']} are kept. "
      f"Selection stops before the formatted context would exceed {meta['optimized']['compression_token_target']} Gemma tokens.\n"
      "5. Kept sentences are written out **verbatim**, in their original reading order, under their source chunk ID. The procedure is deterministic, and the selected segment IDs and scores are stored in `selected_segments`.\n\n"
      f"Parameter choice: {meta['optimized']['compression_parameters_chosen']}. "
      f"**Observation:** the {meta['optimized']['compression_token_target']}-token budget was reached in **{ana['compression_budget_hit_runs']}** optimized runs. "
      f"The mean top-3 pre-compression context was {fs('pre_compression_context_tokens', 'optimized_mean')} tokens, so the reduction came from the relevance threshold and minimum-segment rule rather than from the token target.\n")

    w("## 9. Evaluation methodology\n")
    w(f"**Tokens.** tiktoken was checked and cannot map this model ({meta['token_counting']['tiktoken_check']}), so it was **not** used. "
      f"Context tokens were counted with the {meta['token_counting']['local_tokenizer']} tokenizer. Prompt and output tokens are the LLM server's own `usage` values from the loaded GGUF tokenizer. "
      f"Local and server prompt counts differed by mean {ana['server_minus_local_prompt_tokens']['mean']}, min {ana['server_minus_local_prompt_tokens']['min']} and max {ana['server_minus_local_prompt_tokens']['max']} tokens over {ana['successful']} runs.")
    if pre:
        w(f" Preflight live cross-check: {pre['token_crosscheck']}.")
    w(f"\n\n**Latency** was measured with `time.perf_counter()` on the same machine that runs the LLM server ({rt_name}):\n")
    for k, v in meta["latency_definitions"].items():
        if k != "timer":
            w(f"- `{k}`: {v}")
    w(f"\n**Statistics.** For each metric the report gives the mean and median over the 24 per-question values, the percentage reduction ((baseline − optimized) / baseline × 100), "
      f"and an exploratory paired Wilcoxon signed-rank p-value on per-question values (scipy {ana['analysis_environment']['scipy']}). "
      "When the baseline value is 0 by design (no reranking or compression stage), no percentage is given. "
      "A negative reduction is reported as an increase, never as an improvement.\n\n"
      "**Answer quality** is assessed manually with the 1–5 rubric, a groundedness label and a correct-refusal label (`manual_evaluation_template.csv`). "
      "Two automated diagnostics are also reported, and **neither is a quality score**:\n"
      "(a) exact-refusal string match, whether a citation is present, and whether every cited chunk ID is one that was actually given to the model;\n"
      "(b) *evidence coverage*: whether verbatim source phrases supporting each reference answer were present in the context sent to the model.\n\n"
      "**Testing.** Before any measurement, offline logic tests and integration tests with the real models were run (Section 15).\n")

    w("## 10. Actual measured results\n")
    w(f"- Pipeline runs: **{ana['rows']}** ({n_runs} runs × 24 questions × 2 pipelines). Successful: **{ana['successful']}**. Failed: **{ana['failed']}**. Transient-error retries: **{ana['retries']}**.")
    w(f"- Token counts came from the LLM server's own `usage` field for {ana['token_count_sources'].get('lm_studio_usage', 0)} of {ana['successful']} runs. Data-validation issues: {ana['validation_issues'] or 'none'}.\n")
    w("### 10.1 Summary (per-question means over runs)\n")
    w("| Metric | Baseline mean | Baseline median | Optimized mean | Optimized median | % reduction (mean) | % reduction (median) | Interpretation | Wilcoxon p |")
    w("|---|---|---|---|---|---|---|---|---|")
    for m in ["raw_retrieved_context_tokens", "pre_compression_context_tokens", "final_context_tokens", "input_prompt_tokens", "output_tokens", "total_tokens"]:
        w(row(m, 1))
    for m in ["retrieval_time_seconds", "reranking_time_seconds", "compression_time_seconds", "ttft_seconds", "generation_time_seconds",
              "total_latency_seconds", "tokens_per_second", "derived_decode_time_seconds", "derived_decode_tokens_per_second", "derived_prefill_tokens_per_second"]:
        w(row(m, 3))
    w("\n*`raw_retrieved_context_tokens` for the optimized pipeline counts all 10 FAISS candidates; that text goes only to the reranker, never to the LLM. "
      "`tokens_per_second` = output tokens ÷ generation time, which includes prompt processing. Decode time = generation − TTFT; prefill rate = prompt tokens ÷ TTFT.*\n")
    w("### 10.2 Each run separately\n")
    w("| Run | Metric | Baseline mean | Optimized mean | % reduction (mean) | % reduction (median) |\n|---|---|---|---|---|---|")
    for _, r in sbr.iterrows():
        if r.metric in ("final_context_tokens", "total_tokens", "ttft_seconds", "total_latency_seconds"):
            w(f"| {int(r.run_id)} | `{r.metric}` | {_fmt(r.baseline_mean)} | {_fmt(r.optimized_mean)} | {_fmt(r.pct_reduction_mean, 1)} | {_fmt(r.pct_reduction_median, 1)} |")
    w("")
    w("### 10.3 Reproducibility across runs\n")
    w("| | Baseline | Optimized |\n|---|---|---|")
    for k, lab in [("answers_identical_across_runs", "Questions whose answer was identical in every run"),
                   ("contexts_identical_across_runs", "Questions whose context was identical in every run"),
                   ("output_tokens_identical_across_runs", "Questions whose output-token count was identical in every run")]:
        w(f"| {lab} | {rep_s['baseline'][k]}/{rep_s['baseline']['pairs']} | {rep_s['optimized'][k]}/{rep_s['optimized']['pairs']} |")
    w(f"| Median coefficient of variation of total latency across runs | {_fmt(rep_s['baseline']['median_latency_cv'], 3)} | {_fmt(rep_s['optimized']['median_latency_cv'], 3)} |\n")
    w("Per-question detail is in `reproducibility_per_question.csv`; run-to-run latency is shown in `latency_per_question_runs.png`.\n")

    w("## 11. Token comparison\n")
    w(f"The experiment observed a mean final context of {fs('final_context_tokens', 'baseline_mean')} tokens for the baseline and {fs('final_context_tokens', 'optimized_mean')} tokens for the optimized pipeline, "
      f"a **{fs('final_context_tokens', 'pct_reduction_mean')}% reduction** (median {fs('final_context_tokens', 'pct_reduction_median')}%). "
      f"Prompt tokens were reduced by {fs('input_prompt_tokens', 'pct_reduction_mean')}% and total tokens by {fs('total_tokens', 'pct_reduction_mean')}%. "
      f"Two steps produced the context reduction: reranking to the top 3 chunks brought the mean from {fs('pre_compression_context_tokens', 'baseline_mean')} to {fs('pre_compression_context_tokens', 'optimized_mean')} tokens, "
      f"and sentence compression brought it to {fs('final_context_tokens', 'optimized_mean')}. "
      f"Output tokens were also lower ({fs('output_tokens', 'baseline_mean')} vs {fs('output_tokens', 'optimized_mean')}); the optimized answer was shorter for {lat['questions_optimized_fewer_output_tokens']}/24 questions. "
      "Shorter answers are not automatically better, because some are incomplete (Section 14).\n")
    w("Charts: `average_final_context_tokens.png`, `average_total_tokens.png`, `per_question_token_comparison.png`.\n")

    w("## 12. Latency comparison\n")
    dd = lat["paired_mean_latency_difference_s(baseline-optimized)"]
    ov = lat["optimized_pre_llm_overhead_s(retrieval+rerank+compress)"]
    tf, dc = lat["ttft_vs_input_prompt_tokens"], lat["decode_time_vs_output_tokens"]
    w(f"Mean total latency was {fs('total_latency_seconds', 'baseline_mean', 2)} s for the baseline and {fs('total_latency_seconds', 'optimized_mean', 2)} s for the optimized pipeline, "
      f"a {fs('total_latency_seconds', 'pct_reduction_mean')}% reduction. The medians were {fs('total_latency_seconds', 'baseline_median', 2)} s and {fs('total_latency_seconds', 'optimized_median', 2)} s ({fs('total_latency_seconds', 'pct_reduction_median')}%). "
      f"Using per-question means, the optimized pipeline was faster on **{lat['questions_optimized_lower_mean_latency']}/{lat['paired_questions']}** questions "
      f"(paired difference median {_fmt(dd['median'])} s, range {_fmt(dd['min'])} to {_fmt(dd['max'])} s). "
      f"Across individual runs it was faster in {lat['run_level_pairs_optimized_faster']}/{lat['run_level_pairs']} question-run pairs.\n")
    w(f"- **Extra cost of the optimized pipeline:** reranking took {fs('reranking_time_seconds', 'optimized_mean', 3)} s and compression {fs('compression_time_seconds', 'optimized_mean', 3)} s on average. "
      f"Together with retrieval, the stages before the LLM averaged {_fmt(ov['mean'], 3)} s (max {_fmt(ov['max'], 3)} s).")
    w(f"- **Prompt processing (TTFT):** {fs('ttft_seconds', 'baseline_mean', 2)} s vs {fs('ttft_seconds', 'optimized_mean', 2)} s; lower for the optimized pipeline on {lat['ttft_paired_difference_s(baseline-optimized)']['questions_optimized_lower']}/24 questions. "
      f"Pooled fit: TTFT ≈ {_fmt(tf['intercept_s'])} s + {_fmt(tf['slope_s_per_token'] * 1000, 1)} ms × prompt tokens, R² = {_fmt(tf['r2'], 3)}.")
    w(f"- **Answer writing (decode):** ≈ {_fmt(dc['slope_s_per_token'], 3)} s per output token (R² = {_fmt(dc['r2'], 3)}), i.e. about {_fmt(1 / dc['slope_s_per_token'], 1)} tokens/s on this hardware. "
      "Because the optimized answers were shorter, part of the generation-time reduction comes from answer length rather than context size. This design cannot fully separate the two effects.\n")
    w("Charts: `average_latency.png`, `token_latency_scatter.png`, `latency_components.png`, `latency_per_question_runs.png`.\n")

    w("## 13. Answer-quality evaluation\n")
    w("**Manual scoring is required and is not part of this report.** `manual_evaluation_template.csv` contains, for each question, the reference answer, both answers from the first run, both contexts, and whether each answer was identical across runs. "
      "The scoring columns are blank on purpose.\n\nAutomated diagnostics (not quality scores); run-by-run values are separated by commas:\n")
    w("| Check | Baseline | Optimized |\n|---|---|---|")
    for col, lab, tot in [("unanswerable_refused_exact", "Unanswerable questions refused with the exact sentence", "unanswerable_total"),
                          ("answerable_refused_exact", "Answerable questions refused (false refusals)", "answerable_total"),
                          ("answers_with_citation", "Answers with ≥ 1 chunk-ID citation (of 24)", None),
                          ("answers_with_invalid_citation_ids", "Answers citing an ID that was **not** in the supplied context", None)]:
        tb = f" (of {int(chk[chk.approach_name == 'baseline'][tot].iloc[0])})" if tot else ""
        w(f"| {lab}{tb} | {per_run('baseline', col)} | {per_run('optimized', col)} |")
    w(f"| Evidence phrases present in the context sent to the LLM (of {ev.loc['baseline', 'evidence_items']}) | {ev.loc['baseline', 'evidence_items_in_final_context']} | {ev.loc['optimized', 'evidence_items_in_final_context']} |")
    w(f"| Evidence phrases in the top-3 chunks before compression | — | {ev.loc['optimized', 'evidence_items_in_pre_compression_context']} |")
    w(f"| Answerable questions with **all** evidence phrases in the context | {ev.loc['baseline', 'questions_with_all_evidence_in_final_context']}/20 | {ev.loc['optimized', 'questions_with_all_evidence_in_final_context']}/20 |\n")
    lost_r = int(ev.loc['baseline', 'evidence_items_in_final_context'] - ev.loc['optimized', 'evidence_items_in_pre_compression_context'])
    lost_c = int(ev.loc['optimized', 'evidence_items_in_pre_compression_context'] - ev.loc['optimized', 'evidence_items_in_final_context'])
    w(f"Relative to the baseline, the optimized pipeline lost {lost_r} evidence phrase(s) at the reranking (top-3) step and a further {lost_c} at the compression step. "
      "Evidence being present does not guarantee a correct answer, and missing evidence does not rule one out. "
      "The evidence phrases are listed in the notebook; the chart is `evidence_coverage.png`.\n")

    w("## 14. Failure cases\n")
    w("### 14.1 Automatically detected (objective criteria, first run)\n")
    ft = failure_table(R)
    if len(ft):
        w("| Question | Pipeline | Detected issue(s) | Recorded answer |\n|---|---|---|---|")
        for _, r in ft.iterrows():
            w(f"| {r['question_id']} | {r['approach']} | {r['issues']} | {r['answer'][:300]}{'…' if len(r['answer']) > 300 else ''} |")
    else:
        w("No case met the automatic criteria.")
    w("\nThe criteria are: false refusal, unanswerable question not refused, a citation to an ID outside the context, no citation, and evidence missing from the context.\n")
    w("### 14.2 Author's qualitative notes\n")
    if notes_path.exists():
        w(notes_path.read_text(encoding="utf-8").strip() + "\n")
    else:
        w("*Pending: to be written after reading the recorded answers of this run (`results/author_notes.md`).*\n")

    w("## 15. Tests performed\n")
    if otest:
        w(f"- **Offline logic tests** ({len(otest)} groups, all passed; stand-in models, no LLM): " + ", ".join(f"`{k}`" for k in otest) + ".")
    if itest:
        w(f"- **Integration tests with the real models** (all passed): embeddings are deterministic and normalised; FAISS self-retrieval {itest['faiss_self_retrieval']}; "
          f"reranker is deterministic; top-5 = first 5 of top-10; compression is verbatim, deterministic and within budget; tokenizer sanity check.")
    w("- **Smoke test:** 2 questions × 2 pipelines followed by an automatic gate (row count, no errors, token and latency fields > 0, server usage present, non-empty answers, valid citations, 5 vs 3 final chunks); the gate passed before the full run started (`results/smoke_test/`).")
    w(f"- **Post-run data validation:** duplicate and missing runs, a single model ID, no missing measurements. Issues found: {ana['validation_issues'] or 'none'}.\n")

    rep_path = R / "replication_vs_pilot.json"
    w("## 16. Limitations\n")
    lims = [
        f"**Small sample.** There are 24 questions and {n_runs} run(s) each. The Wilcoxon p-values are exploratory and do not support claims beyond this dataset, model and machine.",
        "**Answer length is a confound.** Generation time is dominated by decoding, and the optimized pipeline often gave shorter answers, partly because some were incomplete or refusals, so not all of the latency gain comes from the smaller context.",
        "**Small synthetic corpus.** The six short documents (45 chunks) and the questions were written by the same author. A real, larger or noisier corpus may behave differently.",
        "**The compression budget was not binding.** The 700-token target was never reached; the relevance threshold (logit ≥ 0) and minimum-5 rule, fixed in advance and not tuned, determined the context size.",
        "**Sentence-level selection can split structure.** Numbered steps and label/value lines can be separated from the text that gives them meaning, and the cross-encoder (trained on MS MARCO web passages) scores isolated list items low.",
        "**Chunk boundaries cut sentences.** With 500 characters and 80 overlap, a chunk can end mid-sentence and give the model a partial fact.",
        ("**Environment unknowns.** GPU offload in LM Studio and whether it honours `seed` are unknown. The local encoders share the CPU with LM Studio, which is intended, since their cost is part of the pipeline." if is_lms else
         f"**Different runtime from the pilot.** These runs used {rt_name} on Colab ({gpu_str}); the pilot used LM Studio on a Windows laptop (CPU/integrated GPU). The model file, prompts and all pipeline settings are the same, but latency is **not** comparable between the two environments, and outputs may differ slightly because runtime builds and chat-template handling differ. Colab is a shared cloud machine, so run-to-run timing can vary."),
        "**Evidence coverage is author-defined.** It is a diagnostic based on author-defined phrases, not a quality score.",
        "**Manual quality scoring is outstanding.** Until it is completed, no claim can be made that answer quality was maintained.",
    ]
    for l in lims:
        w(f"- {l}")
    w("")

    w("## 17. Discussion\n")
    w(f"The measured results show a large reduction in tokens sent to the LLM (context −{fs('final_context_tokens', 'pct_reduction_mean')}%, prompt −{fs('input_prompt_tokens', 'pct_reduction_mean')}%), "
      f"and a lower end-to-end latency on {lat['questions_optimized_lower_mean_latency']}/24 questions. "
      f"The extra local stages (≈{_fmt(ov['mean'], 2)} s) should be weighed against LLM time on this hardware ({_fmt(s.loc['generation_time_seconds', 'optimized_mean'], 2)} s mean generation for the optimized pipeline). The faster the LLM, the larger the share taken by this fixed cost. "
      f"The diagnostics point to a possible quality cost, which the manual scoring must confirm. False refusals on answerable questions, per run: baseline {per_run('baseline', 'answerable_refused_exact')}; optimized {per_run('optimized', 'answerable_refused_exact')}. "
      f"All evidence was present for {ev.loc['optimized', 'questions_with_all_evidence_in_final_context']}/20 answerable questions with the optimized pipeline, against {ev.loc['baseline', 'questions_with_all_evidence_in_final_context']}/20 with the baseline.")
    if rep_path.exists():
        rj_ = json.loads(rep_path.read_text(encoding="utf-8"))
        w(f" **Replication:** compared with the earlier pilot run ({PILOT_DIR_NAME}; LM Studio on the laptop), {rj_['answers_identical_to_pilot']}/{rj_['pairs_compared']} current answers and {rj_['contexts_identical_to_pilot']}/{rj_['pairs_compared']} contexts were identical, and the token reductions can be compared in `replication_vs_pilot.csv`." + ("" if is_lms else " Latency values are not comparable across the two environments."))
    w("\n")

    w("## 18. Conclusion\n")
    w(f"**Observed result.** Across {n_runs} run(s) of 24 questions (Gemma 3 4B Q4_K_M, {where}), the token-efficient pipeline "
      f"reduced mean final context tokens by {fs('final_context_tokens', 'pct_reduction_mean')}%, total tokens by {fs('total_tokens', 'pct_reduction_mean')}%, "
      f"and mean end-to-end latency by {fs('total_latency_seconds', 'pct_reduction_mean')}% (median {fs('total_latency_seconds', 'pct_reduction_median')}%). "
      f"Reranking and compression added about {_fmt(ov['mean'], 2)} s per question. "
      f"False refusals on answerable questions: baseline {per_run('baseline', 'answerable_refused_exact')}, optimized {per_run('optimized', 'answerable_refused_exact')} (per run). "
      f"Complete supporting evidence reached the model for {ev.loc['optimized', 'questions_with_all_evidence_in_final_context']} vs {ev.loc['baseline', 'questions_with_all_evidence_in_final_context']} of 20 answerable questions.\n\n"
      "**Interpretation.** The measured results indicate that, in this setting, reranking plus extractive compression can substantially reduce context-token usage and latency. "
      "Whether answer quality was *maintained* cannot be concluded from these automated diagnostics; manual rubric scoring is required. The diagnostics point to losses on list-structured and multi-step questions.\n\n"
      "**Limitation.** The findings come from a small synthetic corpus on one machine with compression parameters that were not tuned, and part of the latency gain is confounded with shorter outputs (Section 16).\n")

    w("## Files\n")
    w("`experiment_results.csv` (one row per run × question × pipeline) · `experiment_details.jsonl` (plus all scored sentences) · `experiment_metadata.json` · "
      "`summary_metrics.csv` · `summary_metrics_by_run.csv` · `per_question_means.csv` · `reproducibility_per_question.csv` · `automated_checks_*.csv` · "
      "`evidence_coverage*.csv` · `latency_decomposition.json` · `analysis_checks.json` · `preflight.json` · `integration_tests.json` · `offline_tests.json` · "
      "`manual_evaluation_template.csv` · `run_log.txt` · `warmup_log.json` · charts (`*.png`) · `smoke_test/` · "
      f"`{PILOT_DIR_NAME}/` (earlier pilot run, unchanged) · `replication_vs_pilot.*`\n")

    (R / "experiment_report.md").write_text("\n".join(T), encoding="utf-8")
    print(f"report written: {R / 'experiment_report.md'} ({len(chr(10).join(T)):,} characters)")

## 17. Offline test suite
**Stand-in** models (a word-count tokenizer, a word-overlap reranker, a random-vector embedder and a scripted LLM client) exercise the logic without downloads or LLM calls.
These stand-ins are never used to produce results.

| Test group | What it verifies |
|---|---|
| `test_dataset` | 6 mapped documents; 24 unique questions; 20 answerable and 4 unanswerable; category counts 8/6/6/4; exact refusal reference; cross-document questions cite ≥ 2 documents |
| `test_unanswerable_topics_absent` | pets, jury duty, salary and parking do not appear anywhere in the corpus |
| `test_chunking` | deterministic; unique IDs; ≤ 500 characters; exact offsets; no text lost; 0 < overlap ≤ 80 |
| `test_split_text_edge_cases` | empty text, short text, invalid overlap rejected |
| `test_split_segments` | list and heading markers stay attached; segments are verbatim |
| `test_compression` | verbatim; deterministic; reading order kept; chunk IDs kept; minimum-segments rule; threshold rule; token budget respected and flagged; overlap duplicates removed |
| `test_citations` | citation parsing handles lists, repeats and non-ID brackets |
| `test_prompt_identical` | exact system prompt, exact template, temperature 0 |
| `test_pipelines` | 5 vs 3 chunks; top-5 ⊂ top-10; token arithmetic; non-negative timings; fallback when server usage is missing |
| `test_retry_logic` | a transient error is retried and recorded; persistent errors fail after the retry limit; non-transient errors are not retried |
| `test_percentages_and_validation` | % reduction formula, negative values and zero division; duplicate and missing-run detection |
| `test_evidence_strings_verbatim` | every evidence phrase appears verbatim in the documents |

In [ ]:
import re
from types import SimpleNamespace as NS

# ---------------- stand-ins (used ONLY by these tests; never used for results) ----------------
class FakeTok:
    name = "fake-whitespace-tokenizer"
    def count(self, t): return len(t.split())
    def count_chat(self, m): return sum(len(x["content"].split()) for x in m) + 10


class FakeCE:
    """Word-overlap scorer: (shared words - 2). Deterministic."""
    def predict(self, pairs, show_progress_bar=False):
        return np.array([len(set(re.findall(r"\w+", q.lower())) & set(re.findall(r"\w+", t.lower()))) - 2.0
                         for q, t in pairs])


class ConstCE:
    def __init__(self, v): self.v = v
    def predict(self, pairs, show_progress_bar=False): return np.full(len(pairs), self.v, dtype=float)


class FakeEmbedder:
    def encode(self, texts, **kw):
        rng = [np.random.default_rng(sum(map(ord, t)) + len(t)).normal(size=16) for t in texts]
        v = np.stack(rng).astype("float32")
        return v / np.linalg.norm(v, axis=1, keepdims=True)


def fake_client(with_usage=True, fail_times=0, exc=None):
    state = {"calls": 0}

    def create(**kw):
        state["calls"] += 1
        if state["calls"] <= fail_times:
            raise exc
        usage = NS(prompt_tokens=100, completion_tokens=7, total_tokens=107) if with_usage else None
        def gen():
            yield NS(choices=[NS(delta=NS(content="Answer text [IT-C05]"), finish_reason=None)], usage=None)
            yield NS(choices=[NS(delta=NS(content=None), finish_reason="stop")], usage=None)
            yield NS(choices=[], usage=usage)
        return gen()
    return NS(chat=NS(completions=NS(create=create)), state=state)


def _fake_rag(chunks, client=None):
    import faiss
    emb = FakeEmbedder()
    vecs = emb.encode([c.text for c in chunks])
    idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    return RAGSystem(emb, idx, chunks, FakeCE(), FakeTok(), client or fake_client())


# ---------------- tests ----------------
def test_dataset():
    qs = json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))
    docs = sorted(p.name for p in config.DOCS_DIR.glob("*.md"))
    assert docs == sorted(config.DOC_ID_MAP), "every document needs a doc-ID mapping (and vice versa)"
    assert len(qs) == 24 and len({q["question_id"] for q in qs}) == 24
    assert sum(q["answerable"] for q in qs) == 20
    cats = {c: sum(q["category"] == c for q in qs) for c in {q["category"] for q in qs}}
    assert cats == {"direct_lookup": 8, "multi_step_procedure": 6, "cross_document": 6, "unanswerable": 4}, cats
    for q in qs:
        if q["answerable"]:
            assert q["expected_documents"] and set(q["expected_documents"]) <= set(config.DOC_ID_MAP.values())
            assert q["category"] != "unanswerable"
        else:
            assert q["reference_answer"] == config.REFUSAL_TEXT and q["category"] == "unanswerable"
    assert all(len(q["expected_documents"]) >= 2 for q in qs if q["category"] == "cross_document")


def test_unanswerable_topics_absent():
    text = " ".join(p.read_text(encoding="utf-8").lower() for p in config.DOCS_DIR.glob("*.md"))
    for q, pat in {"Q21": r"\bpets?\b", "Q22": r"\bjury\b", "Q23": r"\bsalar(y|ies)\b", "Q24": r"\bparking\b"}.items():
        assert not re.search(pat, text), f"{q}: topic {pat} appears in the documents"


def test_chunking():
    chunks = load_and_chunk()
    assert [c.to_dict() for c in chunks] == [c.to_dict() for c in load_and_chunk()], "chunking not deterministic"
    docs = {p.name: p.read_text(encoding="utf-8") for p in config.DOCS_DIR.glob("*.md")}
    assert len({c.chunk_id for c in chunks}) == len(chunks)
    for c in chunks:
        assert re.fullmatch(r"[A-Z]+-C\d{2}", c.chunk_id)
        assert 0 < len(c.text) <= config.CHUNK_SIZE
        assert docs[c.doc_name][c.char_start:c.char_end] == c.text, "provenance offsets wrong"
    by = {}
    for c in chunks:
        by.setdefault(c.doc_name, []).append(c)
    for name, lst in by.items():
        cov = np.zeros(len(docs[name]), bool)
        for c in lst:
            cov[c.char_start:c.char_end] = True
        assert all(cov[i] or docs[name][i].isspace() for i in range(len(cov))), f"text lost in {name}"
        for a, b in zip(lst, lst[1:]):
            ov = a.char_end - b.char_start
            assert 0 < ov <= config.CHUNK_OVERLAP, (a.chunk_id, b.chunk_id, ov)
    return chunks


def test_split_text_edge_cases():
    assert split_text("", 500, 80) == []
    assert split_text("short text", 500, 80) == [(0, 10)]
    try:
        split_text("abc", 50, 50)
        raise AssertionError("overlap >= chunk size must be rejected")
    except AssertionError as e:
        if "must be rejected" in str(e):
            raise


def test_split_segments():
    t = "## 7. POS Terminal Failure Procedure\nIf a POS fails. Do X.\n1. Restart the terminal and wait 2 minutes.\n4. Log a ticket. Use Priority 1."
    segs = split_segments(t)
    assert segs[0] == "## 7. POS Terminal Failure Procedure"
    assert "1. Restart the terminal and wait 2 minutes." in segs, "list marker must stay attached"
    assert "4. Log a ticket." in segs and "Use Priority 1." in segs
    assert all(s in t for s in segs), "segments must be verbatim substrings"


def test_compression(chunks):
    tok = FakeTok()
    top = [c for c in chunks if c.doc_id == "IT"][5:8]
    q = "What steps when a POS terminal fails?"
    ctx, allsegs, sel, hit = compress(q, top, FakeCE(), tok)
    assert sel and all(s.text in top[s.chunk_rank].text for s in sel), "not verbatim"
    assert ctx == compress(q, top, FakeCE(), tok)[0], "not deterministic"
    assert [(s.chunk_rank, s.position) for s in sel] == sorted((s.chunk_rank, s.position) for s in sel), "reading order"
    for cid in {s.chunk_id for s in sel}:
        assert f"[{cid}]" in ctx, "source chunk ID lost"
    # all scores below threshold -> exactly COMPRESSION_MIN_SEGMENTS segments kept
    _, _, sel_neg, _ = compress(q, top, ConstCE(-5.0), tok)
    assert len(sel_neg) == config.COMPRESSION_MIN_SEGMENTS
    # all scores above threshold and a large budget -> every candidate kept
    _, all_pos, sel_pos, hit_pos = compress(q, top, ConstCE(5.0), tok, token_target=10_000)
    assert len(sel_pos) == len(all_pos) and not hit_pos
    # tiny budget -> budget respected and flagged
    ctx_b, _, _, hit_b = compress(q, top, ConstCE(5.0), tok, token_target=25)
    assert hit_b and tok.count(ctx_b) <= 25
    # overlap duplicates are removed
    A = NS(chunk_id="X-C00", text="Alpha sentence one. Shared sentence here.")
    B = NS(chunk_id="X-C01", text="Shared sentence here. Beta sentence two.")
    _, segs, _, _ = compress("shared sentence", [A, B], ConstCE(1.0), tok)
    assert sum(s.text == "Shared sentence here." for s in segs) == 1


def test_citations():
    assert extract_cited_ids("x [IT-C05, SECURITY-C02] y [LEAVE-C01]") == ["IT-C05", "SECURITY-C02", "LEAVE-C01"]
    assert extract_cited_ids("a [IT-C05][IT-C05] b") == ["IT-C05"]
    assert extract_cited_ids("see [1] and [note]") == []
    assert extract_cited_ids(None) == [] and extract_cited_ids("none") == []


def test_prompt_identical():
    assert config.SYSTEM_PROMPT == ("Answer only from the provided context. Do not use outside knowledge. If the answer "
                                    "is not contained in the context, say exactly: 'I cannot find the answer in the provided "
                                    "documents.' Cite the relevant source chunk IDs in square brackets at the end of each answer.")
    m = build_messages("CTX", "Q?")
    assert m[0] == {"role": "system", "content": config.SYSTEM_PROMPT}
    assert m[1]["content"] == "Context:\nCTX\n\nQuestion: Q?"
    assert config.GENERATION_PARAMS["temperature"] == 0.0


def test_pipelines(chunks):
    rag = _fake_rag(chunks)
    b, o = rag.run_baseline("How many leave days?"), rag.run_optimized("How many leave days?")
    assert json.loads(b["retrieved_chunks"]) == json.loads(o["retrieved_chunks"])[:5], "same index/query embedding"
    assert len(json.loads(b["final_chunks"])) == 5 and len(json.loads(o["final_chunks"])) == 3
    assert set(json.loads(o["final_chunks"])) <= set(json.loads(o["retrieved_chunks"]))
    assert b["system_prompt"] == o["system_prompt"] == config.SYSTEM_PROMPT
    for r in (b, o):
        assert r["input_prompt_tokens"] == 100 and r["output_tokens"] == 7 and r["total_tokens"] == 107
        assert r["token_count_source"] == "lm_studio_usage"
        for k in ["retrieval_time_seconds", "reranking_time_seconds", "compression_time_seconds",
                  "generation_time_seconds", "total_latency_seconds"]:
            assert r[k] >= 0
        assert r["total_latency_seconds"] >= r["generation_time_seconds"]
    assert b["reranking_time_seconds"] == 0 and b["compression_time_seconds"] == 0
    # server usage missing -> falls back to the local tokenizer and says so
    r = _fake_rag(chunks, fake_client(with_usage=False)).run_baseline("How many leave days?")
    assert r["token_count_source"].startswith("local_gemma_tokenizer")


def test_retry_logic():
    import httpx
    import time as _time
    real_sleep, _time.sleep = _time.sleep, (lambda s: None)
    try:
        exc = APIConnectionError(request=httpx.Request("POST", "http://localhost:1234/v1/chat/completions"))
        c = fake_client(fail_times=1, exc=exc)
        out = generate(c, build_messages("c", "q"))
        assert out["retry_count"] == 1 and "APIConnectionError" in out["retry_errors"]
        c = fake_client(fail_times=99, exc=exc)
        try:
            generate(c, build_messages("c", "q")); raise AssertionError("should have failed")
        except RuntimeError:
            assert c.state["calls"] == config.MAX_TRANSIENT_RETRIES + 1
        c = fake_client(fail_times=1, exc=ValueError("bad request"))
        try:
            generate(c, build_messages("c", "q")); raise AssertionError("should have failed")
        except ValueError:
            assert c.state["calls"] == 1, "non-transient errors must not be retried"
    finally:
        _time.sleep = real_sleep


def test_percentages_and_validation():
    assert pct_reduction(100, 50) == 50 and describe(50) == "reduction"
    assert pct_reduction(100, 120) == -20 and describe(-20).startswith("increase")
    assert pct_reduction(0, 5) is None and describe(None).startswith("unavailable")
    assert describe(pct_reduction(100, 100)) == "no change"
    qs = json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))
    rows = [{"run_id": 1, "question_id": q["question_id"], "approach_name": a, "error_message": "", "model_name": "m",
             "final_context_tokens": 1, "input_prompt_tokens": 1, "output_tokens": 1, "total_latency_seconds": 1.0}
            for q in qs for a in APPROACHES]
    df = pd.DataFrame(rows)
    assert validate(df, qs) == []
    issues = validate(pd.concat([df, df.iloc[:1]]), qs)
    assert any("duplicate" in i for i in issues)
    issues = validate(df[df.question_id != "Q05"], qs)
    assert any("Q05" in i for i in issues)


def test_evidence_strings_verbatim():
    docs = _norm_ev(" ".join(p.read_text(encoding="utf-8") for p in config.DOCS_DIR.glob("*.md")))
    qs = {q["question_id"]: q for q in json.loads(config.QUESTIONS_PATH.read_text(encoding="utf-8"))}
    assert set(EVIDENCE) == {k for k, q in qs.items() if q["answerable"]}
    for q, evs in EVIDENCE.items():
        for e in evs:
            assert _norm_ev(e) in docs, (q, e)


def run_offline_tests():
    tests = [test_dataset, test_unanswerable_topics_absent, test_split_text_edge_cases, test_split_segments,
             test_citations, test_prompt_identical, test_retry_logic, test_percentages_and_validation,
             test_evidence_strings_verbatim]
    results = {}
    chunks = test_chunking(); results["test_chunking"] = "passed"
    for t in tests:
        t(); results[t.__name__] = "passed"
    test_compression(chunks); results["test_compression"] = "passed"
    test_pipelines(chunks); results["test_pipelines"] = "passed"
    for k in results:
        print(f"  {k:35s} passed")
    print(f"ALL {len(results)} OFFLINE TEST GROUPS PASSED")
    return results

## 18. Final self-check
After analysis, every deliverable is re-verified:
- all required files and columns are present, with one row per run × question × pipeline and no duplicates;
- one model ID, temperature 0, and an identical system prompt and template in every run;
- no missing measurements, and the token arithmetic is correct;
- the percentage formula recomputes exactly, and no percentage is given for zero-baseline stages;
- the manual scores are blank.

In [ ]:
REQUIRED_COLUMNS = ["question_id", "question", "answerable", "approach_name", "model_name", "embedding_model", "reranker",
                    "retrieved_chunks", "final_chunks", "raw_retrieved_context_tokens", "final_context_tokens",
                    "input_prompt_tokens", "output_tokens", "total_tokens", "retrieval_time_seconds",
                    "reranking_time_seconds", "compression_time_seconds", "generation_time_seconds",
                    "total_latency_seconds", "tokens_per_second", "source_chunk_ids", "generated_answer", "error_message"]
REQUIRED_FILES = ["experiment_results.csv", "experiment_details.jsonl", "experiment_metadata.json", "summary_metrics.csv",
                  "manual_evaluation_template.csv", "experiment_report.md", "average_final_context_tokens.png",
                  "average_total_tokens.png", "average_latency.png", "token_latency_scatter.png",
                  "per_question_token_comparison.png", "evidence_coverage.png", "latency_components.png",
                  "latency_per_question_runs.png"]


def final_self_check(results_dir=None):
    """Re-verifies the finished results. Saves results/final_self_check.json. Returns True if all checks pass."""
    R = Path(results_dir) if results_dir else config.RESULTS_DIR
    checks = {}
    def check(name, cond, detail=""):
        checks[name] = {"passed": bool(cond), "detail": str(detail)}
    for f_ in REQUIRED_FILES:
        check(f"file exists: {f_}", (R / f_).exists())
    raw = pd.read_csv(R / "experiment_results.csv", keep_default_na=False)
    df = load(R)
    meta = json.loads((R / "experiment_metadata.json").read_text(encoding="utf-8"))
    n_runs = int(meta.get("n_runs", 1))
    check("all required CSV columns present", set(REQUIRED_COLUMNS) <= set(raw.columns), set(REQUIRED_COLUMNS) - set(raw.columns))
    ok = df[df.error_message == ""]
    check("rows = runs x 24 questions x 2 pipelines (successful)", len(ok) == n_runs * 48, f"{len(ok)} vs {n_runs * 48}")
    check("no duplicate successful (run, question, pipeline)", not ok.duplicated(["run_id", "question_id", "approach_name"]).any())
    check("24 distinct questions", ok.question_id.nunique() == 24)
    check("failed runs are recorded, not hidden", True, f"{int((df.error_message != '').sum())} failed rows kept in CSV")
    check("single model ID", ok.model_name.nunique() == 1 and ok.model_name.iloc[0] == config.OPENAI_MODEL, ok.model_name.unique())
    check("temperature 0 recorded", meta["llm"]["generation_params"]["temperature"] == 0)
    J = [json.loads(l) for l in open(R / "experiment_details.jsonl", encoding="utf-8")]
    check("identical system prompt in every run", {j.get("system_prompt") for j in J if not j.get("error_message")} == {config.SYSTEM_PROMPT})
    check("user message = template(context, question) in every run",
          all(j["user_message"] == config.USER_TEMPLATE.format(context=j["final_context"], question=j["question"]) for j in J if not j.get("error_message")))
    check("no missing measurements", not ok[["final_context_tokens", "input_prompt_tokens", "output_tokens", "total_tokens",
                                             "generation_time_seconds", "total_latency_seconds"]].isna().any().any())
    check("total_tokens = prompt + output", (ok.total_tokens == ok.input_prompt_tokens + ok.output_tokens).all())
    check("baseline sends 5 chunks, optimized 3", all(len(json.loads(r.final_chunks)) == (5 if r.approach_name == "baseline" else 3) for _, r in ok.iterrows()))
    check("source IDs parsed for every answer", ok.source_chunk_ids.notna().all())
    s = pd.read_csv(R / "summary_metrics.csv").set_index("metric")
    pq = per_question(df)
    b = pq[pq.approach_name == "baseline"].final_context_tokens.mean(); o = pq[pq.approach_name == "optimized"].final_context_tokens.mean()
    check("summary % reduction recomputes exactly", abs(s.loc["final_context_tokens", "pct_reduction_mean"] - (b - o) / b * 100) < 1e-9)
    check("no % reduction reported for zero-baseline stages", s.loc[["reranking_time_seconds", "compression_time_seconds"], "pct_reduction_mean"].isna().all())
    m = pd.read_csv(R / "manual_evaluation_template.csv", keep_default_na=False)
    check("manual template has 24 rows", len(m) == 24)
    check("manual scores not auto-filled (blank unless you filled them)", all(m[c].astype(str).str.strip().eq("").all() for c in SCORE_COLUMNS)
          or (R / "manual_evaluation_template_NEW.csv").exists())
    check("warm-up not in results", WARMUP_QUESTION not in set(ok.question))
    (R / "final_self_check.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
    failed = [k for k, v in checks.items() if not v["passed"]]
    for k, v in checks.items():
        print(("PASS " if v["passed"] else "FAIL ") + k + (f"  [{v['detail']}]" if v["detail"] and not v["passed"] else ""))
    print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
    return not failed

## 19. Run
This cell does everything selected by `RUN_MODE`.

In [ ]:
if RUN_OFFLINE_TESTS:
    offline_results = run_offline_tests()
    config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    if RUN_MODE != "analyze_existing":   # keep the record of the tests that ran together with the experiment
        (config.RESULTS_DIR / "offline_tests.json").write_text(json.dumps(offline_results, indent=2), encoding="utf-8")

if RUN_MODE in ("smoke", "full"):
    comps = preflight()
    run_integration_tests(comps)
    run_experiment(smoke=True, comps=comps)
    if not smoke_gate():
        raise RuntimeError("Smoke gate failed - full run NOT started. Inspect results/smoke_test/.")
    if RUN_MODE == "full":
        run_experiment(comps=comps)
elif RUN_MODE == "resume":
    run_experiment(resume=True)
elif RUN_MODE != "analyze_existing":
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

if RUN_MODE != "smoke" and (config.RESULTS_DIR / "experiment_results.csv").exists():
    analysis = analyze_results()
    evidence_coverage()
    latency_decomposition()
    replication_vs_pilot()
    build_report()
    all_ok = final_self_check()
else:
    print("No full-run results to analyse (smoke mode, or results/experiment_results.csv missing).")

## 20. Results and visualisations
All tables and charts below are read from the files written above. They are drawn from `results/experiment_results.csv` and the diagnostics derived from it; no values are hard-coded.
The full write-up is in `results/experiment_report.md`.

In [ ]:
from IPython.display import display, Image, Markdown

R = config.RESULTS_DIR
if (R / "summary_metrics.csv").exists():
    display(Markdown("### Summary metrics (per-question means over runs)"))
    display(pd.read_csv(R / "summary_metrics.csv")[["metric", "baseline_mean", "baseline_median", "optimized_mean",
            "optimized_median", "pct_reduction_mean", "pct_reduction_median", "mean_change_interpretation",
            "wilcoxon_signed_rank_p_paired"]].round(3))
    display(Markdown("### Each run separately"))
    display(pd.read_csv(R / "summary_metrics_by_run.csv").round(3))
    display(Markdown("### Automated checks (not quality scores)"))
    display(pd.read_csv(R / "automated_checks_summary.csv"))
    display(pd.read_csv(R / "evidence_coverage_summary.csv"))
    for png, cap in [("average_final_context_tokens.png", "Context tokens sent to the LLM"),
                     ("average_total_tokens.png", "Total tokens (prompt + output)"),
                     ("per_question_token_comparison.png", "Context tokens per question"),
                     ("average_latency.png", "Latency by pipeline stage"),
                     ("token_latency_scatter.png", "Context size vs latency"),
                     ("latency_components.png", "Prompt processing vs answer writing"),
                     ("latency_per_question_runs.png", "Run-to-run latency variability"),
                     ("evidence_coverage.png", "Evidence reaching the LLM (diagnostic)")]:
        display(Markdown(f"#### {cap}  (`{png}`)"))
        display(Image(filename=str(R / png)))

## 21. Export results (Colab)
In Colab, this cell zips `data/`, `index/` and `results/`, including the llama.cpp logs, into `/content/Token_Efficient_RAG_results.zip`.

In [ ]:
# Package the results for download (Colab only). Creates /content/Token_Efficient_RAG_results.zip
if IN_COLAB:
    import shutil
    for f in ["/content/llama_server.log", "/content/llama_build.log"]:
        if Path(f).exists():
            shutil.copy(f, config.RESULTS_DIR / Path(f).name)
    out_zip = shutil.make_archive("/content/Token_Efficient_RAG_results", "zip", root_dir=str(PROJECT_ROOT),
                                  base_dir=".")
    print(out_zip, f"{Path(out_zip).stat().st_size / 1e6:.2f} MB")
    print("To download: from google.colab import files; files.download('/content/Token_Efficient_RAG_results.zip')")

## 22. Author's notes on failure cases
*To be added after the final run, once the recorded answers have been read. These notes will be the author's observations, not rubric scores.*